In [19]:
"""
SYNTHETIC DATA GENERATOR
========================
Three-tier training data system:

    TIER 1 — Real training pairs        weight=1.0  (sacred)
    TIER 2 — Encyclopedia extracts      weight=0.6  (attested, not parallel)
    TIER 3 — Synthetic generated pairs  weight=0.3  (inferred)

Synthetic pairs built from:
  - Logogram anchors (100% confidence from real data)
  - Vocabulary from encyclopedia extracts (headword + definition)
  - Grammatical patterns mined from real training pairs

OUTPUT: weighted_pairs list of (akk_tokens, eng_string, weight)
        Drop-in for train_pairs. Real data always first, always governs.
"""

import pandas as pd
import numpy as np
import re
import random
from collections import defaultdict

import os
for path in [
    '/kaggle/input/datasets/deeppast/old-assyrian-grammars-and-other-resources/onomasticon.csv',
    '/kaggle/input/competitions/deep-past-initiative-machine-translation/train.csv',
    '/kaggle/input/competitions/deep-past-initiative-machine-translation/test.csv',
]:
    print(f"EXISTS: {os.path.exists(path)} — {path}")

random.seed(42)
np.random.seed(42)

# ── LOGOGRAM ANCHORS (100% confidence) ──────────────────────────────
LOGOGRAM_ANCHORS = {
    'KÙ.BABBAR' : 'silver',
    'KÙ.AN'     : 'tin',
    'AN.NA'     : 'tin',
    'URUDU'     : 'copper',
    'GÍN'       : 'shekel',
    'MA.NA'     : 'mina',
    'GÚ'        : 'talent',
    'DUMU'      : 'son of',
    'DUMU.MUNUS': 'daughter of',
    'DAM'       : 'wife of',
    'DAM.GÀR'   : 'merchant',
    'KIŠIB'     : 'seal of',
    'É'         : 'house',
    'É.GAL'     : 'palace',
    'ANŠE'      : 'donkey',
    'TÚG'       : 'textile',
    'UDU'       : 'sheep',
    'SÍG'       : 'wool',
    'ITU'       : 'month',
    'MU'        : 'year',
    'IGI'       : 'witnessed by',
    'ŠU.NÍGIN'  : 'total',
    'SIG₅'      : 'fine quality',
    'LUGAL'     : 'king',
    'KUR'       : 'land',
}

# ── GRAMMATICAL PATTERNS (mined from real corpus) ────────────────────
# {LOG} = logogram, {LOG_ENG} = its English, {NUM} = number, {PN}/{PN2} = proper noun
PATTERNS = [
    ('KIŠIB {PN}',                          'Seal of {PN}'),
    ('KIŠIB {PN} DUMU {PN2}',              'Seal of {PN} son of {PN2}'),
    ('{NUM} {LOG} {PN} i-dí-in',            '{PN} gave {NUM} {LOG_ENG}'),
    ('{NUM} {LOG} a-na {PN} i-dí-in',       'He gave {NUM} {LOG_ENG} to {PN}'),
    ('{NUM} {LOG} i-lá-qé',                 'He took {NUM} {LOG_ENG}'),
    ('{NUM} MA.NA {LOG} SIG₅',              '{NUM} minas of fine {LOG_ENG}'),
    ('{NUM} GÍN {LOG}',                     '{NUM} shekels of {LOG_ENG}'),
    ('um-ma {PN}-ma a-na {PN2} qí-bi-ma',  'Thus says {PN}: speak to {PN2}'),
    ('um-ma {PN}-ma',                       'Thus says {PN}'),
    ('a-na {PN} qí-bi-ma',                 'Speak to {PN}'),
    ('IGI {PN} DUMU {PN2}',                'Witnessed by {PN} son of {PN2}'),
    ('{PN} DUMU {PN2}',                    '{PN} son of {PN2}'),
    ('{PN} DAM {PN2}',                     '{PN} wife of {PN2}'),
    ('TÚG SIG₅',                           'fine textile'),
    ('{NUM} TÚG',                           '{NUM} textile'),
    ('{NUM} ANŠE',                          '{NUM} donkeys'),
    ('{NUM} UDU',                           '{NUM} sheep'),
]

SAMPLE_PNS = [
    'Puzur-Aššur', 'Šu-Illil', 'Mannum-kī-Aššur', 'Enna-Suen',
    'Aššur-nādā', 'Kūbum', 'Ilī-dan', 'Šalim-Aššur',
    'Innāya', 'Aššur-idi', 'Buzāzu', 'Tarām-Kūbi',
]

SAMPLE_NUMS = ['1', '2', '3', '5', '10', '20', '1.5', '0.5', '½', '⅓']


# ── TIER 2: ENCYCLOPEDIA LOADER ──────────────────────────────────────

def _extract_gloss(definition):
    m = re.search(r'"([^"]{2,40})"', definition)
    if m:
        g = m.group(1).strip()
        if not g.startswith('~') and not g.startswith('('):
            return g.split(',')[0].split(';')[0].strip()
    clean = re.sub(r'\([^)]+\)', '', definition)
    clean = clean.split(';')[0].split(',')[0].strip()
    words = clean.split()
    if 2 <= len(words) <= 5:
        return clean
    return None


def load_encyclopedia_pairs(lex_paths, weight=0.6):
    pairs = []
    for path in lex_paths:
        try:
            df = pd.read_csv(path)
            for _, row in df.iterrows():
                hw   = str(row.get('headword',   '')).strip()
                gram = str(row.get('grammar',    '')).strip()
                defn = str(row.get('definition', '')).strip()
                if not hw or not defn or hw == 'nan':
                    continue
                gloss = _extract_gloss(defn)
                if not gloss:
                    continue
                if gram == 'v.':
                    akk = f'i-{hw[:4]}-um'
                    eng = f'he {gloss}'
                elif gram == 'adj.':
                    akk = f'{hw} GÍN KÙ.BABBAR'
                    eng = f'{gloss} shekel of silver'
                elif gram == 's.':
                    akk = f'1 {hw}'
                    eng = f'one {gloss}'
                else:
                    akk = hw
                    eng = gloss
                pairs.append((akk.split(), eng, weight))
        except Exception as e:
            print(f"[SYNTH] Warning loading {path}: {e}")
    print(f"[SYNTH] Tier 2 (encyclopedia):  {len(pairs)} pairs @ {weight}")
    return pairs


# ── TIER 3: SYNTHETIC GENERATOR ──────────────────────────────────────

def generate_synthetic_pairs(n=500, weight=0.3):
    pairs = []
    logo_items = list(LOGOGRAM_ANCHORS.items())
    for _ in range(n):
        pat_akk, pat_eng = random.choice(PATTERNS)
        pn      = random.choice(SAMPLE_PNS)
        pn2     = random.choice([p for p in SAMPLE_PNS if p != pn])
        num     = random.choice(SAMPLE_NUMS)
        log_akk, log_eng = random.choice(logo_items)

        akk = (pat_akk
               .replace('{PN}',     pn)
               .replace('{PN2}',    pn2)
               .replace('{NUM}',    num)
               .replace('{LOG}',    log_akk)
               .replace('{LOG_ENG}', log_eng))
        eng = (pat_eng
               .replace('{PN}',     pn)
               .replace('{PN2}',    pn2)
               .replace('{NUM}',    num)
               .replace('{LOG}',    log_akk)
               .replace('{LOG_ENG}', log_eng))

        pairs.append((akk.split(), eng, weight))
    print(f"[SYNTH] Tier 3 (synthetic):     {len(pairs)} pairs @ {weight}")
    return pairs


# ── ASSEMBLER ────────────────────────────────────────────────────────

def build_weighted_pairs(train_pairs, lex_paths):
    """
    Returns weighted_pairs: list of (akk_tokens, eng_string, weight)
    Tier 1 always first. Real data governs all conflicts.
    """
    weighted = [(akk, eng, 1.0) for akk, eng in train_pairs]
    print(f"[SYNTH] Tier 1 (real):          {len(train_pairs)} pairs @ 1.0")

    weighted.extend(load_encyclopedia_pairs(lex_paths, weight=0.6))
    weighted.extend(generate_synthetic_pairs(n=500, weight=0.3))

    print(f"[SYNTH] Total weighted pairs:   {len(weighted)}")
    return weighted


# ── HOW TO PLUG IN ───────────────────────────────────────────────────
#
# 1. After building train_pairs in main(), add:
#
#   LEX_PATHS = [
#       '/kaggle/input/.../akkadian_lexiconfromdick.csv',
#       '/kaggle/input/.../akkadian_lexiconfromdick1-74.csv',
#   ]
#   weighted_pairs = build_weighted_pairs(train_pairs, LEX_PATHS)
#
# 2. Where mappings are built, replace:
#   for akk_tokens, eng_sent in train_pairs:
# with:
#   for akk_tokens, eng_sent, w in weighted_pairs:
#
# 3. Weight the counters by w:
#   root_to_eng_words[root][word]          += w   # was += 1
#   akk_cluster_to_eng_words[cluster][word]+= w   # was += 1
#
# 4. SOM training still uses only real token lists:
#   all_akk_tokens = list(set(tok for akk,_,_ in weighted_pairs[:len(train_pairs)] for tok in akk))
#   all_eng_words  = list(set(word for _,eng,_ in weighted_pairs[:len(train_pairs)] for word in eng.split()))
# ─────────────────────────────────────────────────────────────────────

EXISTS: True — /kaggle/input/datasets/deeppast/old-assyrian-grammars-and-other-resources/onomasticon.csv
EXISTS: True — /kaggle/input/competitions/deep-past-initiative-machine-translation/train.csv
EXISTS: True — /kaggle/input/competitions/deep-past-initiative-machine-translation/test.csv


In [20]:
"""
UNIFIED SOM
===========
Single SOM trained on PAIRED feature vectors:
  - Left half:  Akkadian token character-level TF-IDF (what the token looks like)
  - Right half: English co-occurrence context (what English words it appears with)

This means the SOM topology encodes CROSS-LINGUAL similarity directly.
Two cells that are adjacent on the grid contain Akkadian tokens that:
  1. Look similar (share morphology), AND
  2. Translate to similar English words

Path through the graph drops from 4 hops (akk→akk_cluster→eng_cluster→eng)
to 2 hops (akk→unified_cell→eng).

Analogy:
  Dual SOM = two separate seismic surveys of different attributes
  Unified SOM = single multi-attribute co-render (amplitude + impedance)
  Cell adjacency = stratigraphic neighborhood that respects BOTH attributes

Tier handling:
  - Real pairs (weight=1.0) govern the SOM topology
  - Synthetic pairs (weight<1.0) fill coverage gaps AFTER real training
  - Synth-only tokens get mapped to nearest real-trained cell (not retrained)
"""

import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from collections import Counter, defaultdict


def build_unified_som_clusters(weighted_pairs, n_cols=4, n_rows=48,
                                n_iter=3000, random_seed=42):
    """
    Build unified cluster dicts from a single bilingual SOM.

    For each Akkadian token in training data, builds a paired vector:
        [akk_char_tfidf | eng_context_tfidf]

    The SOM learns topology over both simultaneously.

    Parameters
    ----------
    weighted_pairs : list of (akk_tokens, eng_string, weight)
    n_cols, n_rows : SOM grid dimensions
    n_iter         : training iterations
    random_seed    : reproducibility

    Returns
    -------
    akk_token_to_cluster : dict  token -> cluster_id
    eng_word_to_cluster  : dict  word  -> cluster_id
    real_akk_tokens      : set   tokens seen in real data
    real_eng_words       : set   words seen in real data
    """

    # ── Step 1: Collect all tokens and build co-occurrence ────────────
    real_pairs = [(akk, eng) for akk, eng, w in weighted_pairs if w >= 1.0]
    all_pairs  = [(akk, eng) for akk, eng, _ in weighted_pairs]

    # Collect unique tokens from real data (governs SOM)
    real_akk_tokens = set()
    real_eng_words  = set()
    for akk, eng in real_pairs:
        real_akk_tokens.update(akk)
        real_eng_words.update(eng.lower().split())

    # Build token -> english context mapping from ALL pairs (weighted)
    # Each Akkadian token accumulates a bag-of-words of the English
    # sentences it appears in, weighted by tier
    akk_to_eng_context = defaultdict(Counter)
    for akk_tokens, eng_sent, w in weighted_pairs:
        eng_words = eng_sent.lower().split()
        for tok in akk_tokens:
            for ew in eng_words:
                akk_to_eng_context[tok][ew] += w

    # Collect all unique Akkadian tokens and English words
    all_akk = list(real_akk_tokens)  # Train SOM on real tokens
    all_eng = sorted(real_eng_words)  # English vocabulary for context vectors

    print(f"[UNIFIED-SOM] Real Akk tokens: {len(all_akk)} | "
          f"Real Eng words: {len(all_eng)}")

    # ── Step 2: Build paired feature vectors ─────────────────────────
    # Left half: character-level TF-IDF of Akkadian token
    akk_vectorizer = TfidfVectorizer(analyzer='char_wb', ngram_range=(2, 4),
                                      max_features=200)
    akk_tfidf = akk_vectorizer.fit_transform(all_akk).toarray().astype(np.float32)

    # Right half: English context vector (which English words does this
    # token co-occur with, and how strongly?)
    # Build as a dense matrix: rows=akk tokens, cols=eng words
    eng_vocab_idx = {w: i for i, w in enumerate(all_eng)}
    n_eng_feats = len(all_eng)
    eng_context = np.zeros((len(all_akk), n_eng_feats), dtype=np.float32)

    for row_idx, tok in enumerate(all_akk):
        ctx = akk_to_eng_context.get(tok, {})
        for ew, count in ctx.items():
            if ew in eng_vocab_idx:
                eng_context[row_idx, eng_vocab_idx[ew]] = count

    # L2-normalize each row (so morphology and context contribute equally)
    akk_norms = np.linalg.norm(akk_tfidf, axis=1, keepdims=True)
    akk_norms[akk_norms == 0] = 1.0
    akk_tfidf_normed = akk_tfidf / akk_norms

    eng_norms = np.linalg.norm(eng_context, axis=1, keepdims=True)
    eng_norms[eng_norms == 0] = 1.0
    eng_context_normed = eng_context / eng_norms

    # Concatenate: [akk_morphology | eng_context]
    # Weight: give English context slightly more influence (0.6 vs 0.4)
    # because cross-lingual alignment is the primary objective
    AKK_WEIGHT = 0.4
    ENG_WEIGHT = 0.6
    paired_vectors = np.hstack([
        akk_tfidf_normed * AKK_WEIGHT,
        eng_context_normed * ENG_WEIGHT
    ])

    n_features = paired_vectors.shape[1]
    print(f"[UNIFIED-SOM] Paired vector: {akk_tfidf.shape[1]} akk feats + "
          f"{n_eng_feats} eng feats = {n_features} total")

    # ── Step 3: Train unified SOM ────────────────────────────────────

    som = MiniSom(
        n_cols, n_rows, n_features,
        sigma=1.5,
        learning_rate=0.7,
        neighborhood_function='gaussian',
        topology='hexagonal',
        random_seed=random_seed
    )

    som.random_weights_init(paired_vectors)
    som.train(paired_vectors, num_iteration=n_iter, verbose=False)

    # ── Step 4: Map Akkadian tokens to cells ─────────────────────────
    akk_token_to_cluster = {}
    for i, tok in enumerate(all_akk):
        col, row = som.winner(paired_vectors[i])
        flat_id = row * n_cols + col
        akk_token_to_cluster[tok] = flat_id

    # ── Step 5: Map synth-only tokens to nearest real cell ───────────
    synth_only_akk = set()
    for akk_tokens, _, w in weighted_pairs:
        if w < 1.0:
            for tok in akk_tokens:
                if tok not in akk_token_to_cluster:
                    synth_only_akk.add(tok)

    if synth_only_akk:
        synth_list = list(synth_only_akk)
        synth_akk_tfidf = akk_vectorizer.transform(synth_list).toarray().astype(np.float32)
        synth_norms = np.linalg.norm(synth_akk_tfidf, axis=1, keepdims=True)
        synth_norms[synth_norms == 0] = 1.0
        synth_akk_tfidf = synth_akk_tfidf / synth_norms

        synth_eng_ctx = np.zeros((len(synth_list), n_eng_feats), dtype=np.float32)
        for row_idx, tok in enumerate(synth_list):
            ctx = akk_to_eng_context.get(tok, {})
            for ew, count in ctx.items():
                if ew in eng_vocab_idx:
                    synth_eng_ctx[row_idx, eng_vocab_idx[ew]] = count
        synth_eng_norms = np.linalg.norm(synth_eng_ctx, axis=1, keepdims=True)
        synth_eng_norms[synth_eng_norms == 0] = 1.0
        synth_eng_ctx = synth_eng_ctx / synth_eng_norms

        synth_paired = np.hstack([
            synth_akk_tfidf * AKK_WEIGHT,
            synth_eng_ctx * ENG_WEIGHT
        ])
        for i, tok in enumerate(synth_list):
            col, row = som.winner(synth_paired[i])
            flat_id = row * n_cols + col
            akk_token_to_cluster[tok] = flat_id

        print(f"[UNIFIED-SOM] Synth-only Akk tokens mapped: {len(synth_only_akk)}")

    # ── Step 6: Map English words to unified cells ───────────────────
    # Each English word gets assigned to the cell where it has the
    # strongest co-occurrence signal. This is the key difference from
    # dual SOM: English words live on the SAME map as Akkadian tokens.
    eng_word_to_cluster = {}

    # Build: for each cell, accumulate English word weights
    cell_eng_weights = defaultdict(Counter)
    for tok, cell_id in akk_token_to_cluster.items():
        ctx = akk_to_eng_context.get(tok, {})
        for ew, count in ctx.items():
            cell_eng_weights[cell_id][ew] += count

    # For each English word, assign to cell with highest weight
    all_eng_in_context = set()
    for ctx in akk_to_eng_context.values():
        all_eng_in_context.update(ctx.keys())

    for ew in all_eng_in_context:
        best_cell = -1
        best_weight = 0
        for cell_id, counter in cell_eng_weights.items():
            if counter[ew] > best_weight:
                best_weight = counter[ew]
                best_cell = cell_id
        if best_cell != -1:
            eng_word_to_cluster[ew] = best_cell

    # ── Coverage report ──────────────────────────────────────────────
    n_cells = n_cols * n_rows
    occupied_by_akk = len(set(akk_token_to_cluster.values()))
    occupied_by_eng = len(set(eng_word_to_cluster.values()))
    shared_cells = len(set(akk_token_to_cluster.values()) &
                       set(eng_word_to_cluster.values()))

    print(f"\n[UNIFIED-SOM] Grid: {n_cols}x{n_rows} = {n_cells} cells")
    print(f"  Akk tokens mapped:  {len(akk_token_to_cluster)} -> "
          f"{occupied_by_akk} cells")
    print(f"  Eng words mapped:   {len(eng_word_to_cluster)} -> "
          f"{occupied_by_eng} cells")
    print(f"  Shared cells:       {shared_cells} "
          f"(cells containing BOTH Akk and Eng)")
    print(f"  Cross-lingual coverage: {shared_cells/n_cells*100:.1f}%")

    return akk_token_to_cluster, eng_word_to_cluster, real_akk_tokens, real_eng_words, paired_vectors, all_akk, som

In [21]:
"""
STAGE 4 — SAM TERRAIN SEGMENTATION
====================================
Takes the SOM grammar terrain and segments it visually.
No Meta SAM needed — we use watershed segmentation (scipy) which is:
  - Available in Kaggle kernel (no install)
  - Appropriate for terrain topology (designed for this exact problem)
  - Faster than SAM for this grid size (56 cells)

WHAT THIS DOES:
  1. Renders the SOM as a 2D grammar density map (same as Stage 1)
  2. Applies watershed segmentation to find natural grammar zones
  3. Each zone = a set of SOM cells that share grammatical neighborhood
  4. Tokens in the same zone inherit zone-level grammar rules
  5. Zone assignments feed back into compose_sequence as a new signal layer

ANALOGY:
  SOM terrain = seismic amplitude map
  Watershed segments = stratigraphic units
  Zone grammar rules = facies interpretation per unit

OUTPUT:
  cell_to_zone     : dict  (row, col) -> zone_id
  token_to_zone    : dict  token -> zone_id
  zone_grammar     : dict  zone_id -> dominant grammar tag + top English words
"""

import numpy as np
from collections import defaultdict, Counter
from scipy import ndimage
from scipy.ndimage import label


# ── GRAMMAR CLASSIFIER ───────────────────────────────────────────────
# Same as Stage 1 — classifies tokens into grammar categories
GRAMMAR_CATEGORIES = {
    'LOGOGRAM' : {'color': (1.0, 0.8, 0.0), 'z_boost': 3.0},
    'VERB'     : {'color': (0.2, 0.6, 1.0), 'z_boost': 2.0},
    'NOUN'     : {'color': (0.2, 0.9, 0.3), 'z_boost': 1.5},
    'PARTICLE' : {'color': (0.9, 0.3, 0.2), 'z_boost': 1.0},
    'PRONOUN'  : {'color': (0.8, 0.2, 0.8), 'z_boost': 1.2},
    'UNKNOWN'  : {'color': (0.4, 0.4, 0.4), 'z_boost': 0.5},
}

LOGOGRAMS = {
    'KÙ.BABBAR', 'KÙ.AN', 'AN.NA', 'URUDU', 'GÍN', 'MA.NA', 'GÚ',
    'DUMU', 'DUMU.MUNUS', 'DAM', 'DAM.GÀR', 'KIŠIB', 'É', 'É.GAL',
    'ANŠE', 'TÚG', 'UDU', 'SÍG', 'ITU', 'MU', 'IGI', 'ŠU.NÍGIN',
    'SIG₅', 'LUGAL', 'KUR', 'KÙ', 'ŠE', 'ZÌ', 'ÍD',
}

VERB_PREFIXES = ('i-', 'u-', 'ta-', 'a-', 'ni-', 'tu-', 'lu-')
PARTICLE_SET  = {
    'a-na', 'i-na', 'ša', 'ú', 'u', 'lá', 'la', 'lu', 'ki-ma',
    'um-ma', 'ma-la', 'a-ma-kam',
}

def classify_token(tok):
    t = tok.upper()
    if any(logo in t for logo in LOGOGRAMS) or '.' in tok:
        return 'LOGOGRAM'
    tl = tok.lower()
    if tl in PARTICLE_SET:
        
        return 'PARTICLE'
    if any(tl.startswith(p) for p in VERB_PREFIXES):
        return 'VERB'
    if tl.endswith(('-um', '-im', '-am', '-im')):
        return 'NOUN'
    return 'UNKNOWN'


# ── BUILD TERRAIN GRID ───────────────────────────────────────────────

def build_terrain_grid(akk_token_to_cluster, n_cols=4, n_rows=48):
    """
    Build 2D grammar density grid from cluster assignments.

    Returns
    -------
    density_grid  : (n_rows, n_cols) float — token count per cell
    grammar_grid  : (n_rows, n_cols) str   — dominant grammar per cell
    cell_tokens   : dict (row,col) -> list of tokens
    """
    cell_tokens   = defaultdict(list)
    cell_grammar  = defaultdict(Counter)

    for tok, flat_id in akk_token_to_cluster.items():
        row = flat_id // n_cols
        col = flat_id  % n_cols
        if row < n_rows and col < n_cols:
            cell_tokens[(row, col)].append(tok)
            gram = classify_token(tok)
            cell_grammar[(row, col)][gram] += 1

    density_grid = np.zeros((n_rows, n_cols), dtype=float)
    grammar_grid = np.full((n_rows, n_cols), 'UNKNOWN', dtype=object)

    for row in range(n_rows):
        for col in range(n_cols):
            toks = cell_tokens.get((row, col), [])
            density_grid[row, col] = len(toks)
            if toks:
                dominant = cell_grammar[(row, col)].most_common(1)[0][0]
                grammar_grid[row, col] = dominant

    return density_grid, grammar_grid, cell_tokens


# ── WATERSHED SEGMENTATION ───────────────────────────────────────────

def watershed_segment(density_grid, grammar_grid, min_zone_size=2):
    """
    Segments the SOM terrain using watershed on grammar-weighted density.

    Strategy:
      - Weight density by grammar category z_boost
      - Smooth with gaussian filter (terrain-like)
      - Find local maxima as seeds
      - Watershed fills from seeds downward
      - Small zones merged into neighbors

    Returns
    -------
    zone_map : (n_rows, n_cols) int — zone ID per cell (0 = unassigned)
    n_zones  : int
    """
    n_rows, n_cols = density_grid.shape

    # Build z-weighted terrain
    # Use log-normalized density to create natural peaks
    z_grid = np.log1p(density_grid)
    # Add small noise to break flat ties and force multiple seeds
    rng = np.random.default_rng(42)
    z_grid += rng.uniform(0.0, 0.05, z_grid.shape)
    
    # Smooth terrain (like seismic smoothing before picking)
    smoothed = ndimage.gaussian_filter(z_grid, sigma=0.2)

    # Find local maxima as watershed seeds
    # A cell is a local max if it's >= all 8 neighbors
    seeds = np.zeros_like(smoothed, dtype=int)
    seed_id = 1
    for row in range(n_rows):
        for col in range(n_cols):
            val = smoothed[row, col]
            if val == 0:
                continue
            neighbors = []
            for dr in [-1, 0, 1]:
                for dc in [-1, 0, 1]:
                    if dr == 0 and dc == 0:
                        continue
                    nr, nc = row + dr, col + dc
                    if 0 <= nr < n_rows and 0 <= nc < n_cols:
                        neighbors.append(smoothed[nr, nc])
            is_max = neighbors and val >= np.mean(neighbors) * 1.05
            if is_max:
                seeds[row, col] = seed_id
                seed_id += 1

    # If no seeds found, treat whole grid as one zone
    if seed_id == 1:
        return np.ones((n_rows, n_cols), dtype=int), 1

    # Watershed: grow zones from seeds using inverted terrain
    # Use scipy label on thresholded image as approximation
    # (Full watershed requires skimage — unavailable; this is equivalent for small grids)
    zone_map  = np.zeros_like(smoothed, dtype=int)
    threshold = np.percentile(smoothed[smoothed > 0], 30) if np.any(smoothed > 0) else 0

    # Assign seed cells
    zone_map[seeds > 0] = seeds[seeds > 0]

    # Grow zones: iteratively assign unassigned cells to nearest seed zone
    max_iter = n_rows * n_cols
    for _ in range(max_iter):
        changed = False
        for row in range(n_rows):
            for col in range(n_cols):
                if zone_map[row, col] != 0:
                    continue
                if smoothed[row, col] < threshold * 0.1:
                    continue
                # Find neighbor zones
                neighbor_zones = []
                for dr in [-1, 0, 1]:
                    for dc in [-1, 0, 1]:
                        nr, nc = row + dr, col + dc
                        if 0 <= nr < n_rows and 0 <= nc < n_cols:
                            if zone_map[nr, nc] > 0:
                                neighbor_zones.append(zone_map[nr, nc])
                if neighbor_zones:
                    # Assign to most common neighbor zone
                    zone_map[row, col] = Counter(neighbor_zones).most_common(1)[0][0]
                    changed = True
        if not changed:
            break

    # Merge tiny zones into largest neighbor
    zone_sizes = Counter(zone_map[zone_map > 0].flat)
    for row in range(n_rows):
        for col in range(n_cols):
            z = zone_map[row, col]
            if z > 0 and zone_sizes[z] < min_zone_size:
                neighbor_zones = []
                for dr in [-1, 0, 1]:
                    for dc in [-1, 0, 1]:
                        nr, nc = row + dr, col + dc
                        if 0 <= nr < n_rows and 0 <= nc < n_cols:
                            nz = zone_map[nr, nc]
                            if nz > 0 and nz != z:
                                neighbor_zones.append(nz)
                if neighbor_zones:
                    zone_map[row, col] = Counter(neighbor_zones).most_common(1)[0][0]

    # Renumber zones 1..N
    unique_zones = sorted(set(zone_map[zone_map > 0].flat))
    remap = {old: new for new, old in enumerate(unique_zones, start=1)}
    remap[0] = 0
    for row in range(n_rows):
        for col in range(n_cols):
            zone_map[row, col] = remap.get(zone_map[row, col], 0)

    n_zones = len(unique_zones)
    return zone_map, n_zones


# ── ZONE GRAMMAR PROFILER ────────────────────────────────────────────

ZONE_STOPWORDS = {'of', 'the', 'to', 'and', 'in', 'a', 'an'}

def profile_zones(zone_map, cell_tokens, grammar_grid,
                  akk_cluster_to_eng_words, n_cols=7):

 
    """
    For each zone, compute:
      - dominant grammar category
      - top English words from cluster mapping
      - token count
      - cell list
    """

    
    n_rows = zone_map.shape[0]
    zone_cells    = defaultdict(list)
    zone_grammar  = defaultdict(Counter)
    zone_eng      = defaultdict(Counter)

    for row in range(n_rows):
        for col in range(n_cols):
            z = zone_map[row, col]
            if z == 0:
                continue
            zone_cells[z].append((row, col))
            gram = grammar_grid[row, col]
            zone_grammar[z][gram] += len(cell_tokens.get((row, col), []))
            # Pull English words from cluster mapping
            flat_id = row * n_cols + col
            for word, prob in akk_cluster_to_eng_words.get(flat_id, {}).items():
                zone_eng[z][word] += prob

    zone_profiles = {}
    for z in zone_cells:
        dominant_gram = zone_grammar[z].most_common(1)[0][0]
        top_eng       = zone_eng[z].most_common(5)
        token_count   = sum(len(cell_tokens.get(c, [])) for c in zone_cells[z])
        zone_profiles[z] = {
            'dominant_grammar' : dominant_gram,
            'top_english'      : top_eng,
            'token_count'      : token_count,
            'cells'            : zone_cells[z],
            'n_cells'          : len(zone_cells[z]),
        }

    return zone_profiles


# ── TOKEN ZONE MAPPING ───────────────────────────────────────────────

def build_token_to_zone(akk_token_to_cluster, zone_map, n_cols=12):
    """
    Maps each Akkadian token to its zone ID.
    """
    token_to_zone = {}
    for tok, flat_id in akk_token_to_cluster.items():
        row = flat_id // n_cols
        col = flat_id  % n_cols
        if row < zone_map.shape[0] and col < n_cols:
            token_to_zone[tok] = int(zone_map[row, col])
    return token_to_zone


# ── MAIN ASSEMBLER ───────────────────────────────────────────────────

def build_zone_system(akk_token_to_cluster, akk_cluster_to_eng_words,
                      n_cols=4, n_rows=48):
    """
    Full pipeline: terrain → watershed → zone profiles → token mapping.

    Returns
    -------
    token_to_zone  : dict  token -> zone_id
    zone_profiles  : dict  zone_id -> {dominant_grammar, top_english, ...}
    zone_map       : (n_rows, n_cols) int array
    """
    density_grid, grammar_grid, cell_tokens = build_terrain_grid(
        akk_token_to_cluster, n_cols, n_rows
    )

    zone_map, n_zones = watershed_segment(density_grid, grammar_grid)

    zone_profiles = profile_zones(
        zone_map, cell_tokens, grammar_grid,
        akk_cluster_to_eng_words, n_cols
    )

    token_to_zone = build_token_to_zone(akk_token_to_cluster, zone_map, n_cols)

    print(f"[ZONE] Watershed segmentation: {n_zones} zones from {n_cols}x{n_rows} grid")
    for z, p in sorted(zone_profiles.items()):
        top = p['top_english'][:3]
        print(f"  Zone {z:2d}: {p['dominant_grammar']:<10} "
              f"{p['token_count']:5d} tokens | {p['n_cells']} cells | "
              f"top_eng={[w for w,_ in top]}")

    return token_to_zone, zone_profiles, zone_map


# ── HOW TO PLUG IN ───────────────────────────────────────────────────
#
# In main(), AFTER building akk_token_to_cluster and akk_cluster_to_eng_words:
#
#   token_to_zone, zone_profiles, zone_map = build_zone_system(
#       akk_token_to_cluster, akk_cluster_to_eng_words
#   )
#
# Then pass token_to_zone into compose_sequence:
#   output, routes = compose_sequence(
#       akk_tokens, constructions, root_to_eng_words,
#       akk_cluster_to_eng_words, G, bigram_counts,
#       eng_word_to_cluster, akk_token_to_cluster,
#       token_to_zone=token_to_zone,        # ← new
#       zone_profiles=zone_profiles         # ← new
#   )
#
# In compose_sequence, zone info adds a grammar-coherence bonus during
# beam scoring: tokens in the same zone as their neighbors score higher.
# ─────────────────────────────────────────────────────────────────────

In [22]:
import re
import os
import time
import pandas as pd
import numpy as np
from pathlib import Path
from heapq import nlargest
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import KMeans
from collections import Counter, defaultdict
import networkx as nx
from sklearn.metrics.pairwise import cosine_similarity
from scipy.spatial.distance import jaccard
import importlib
import sys
from difflib import get_close_matches
import unicodedata

if 'minisom' in sys.modules:
    del sys.modules['minisom']

# ═══════════════════════════════════════════════════════════════════════
# MINISOM (inline to avoid import issues on Kaggle)
# ═══════════════════════════════════════════════════════════════════════

class MiniSom:
    def __init__(self, x, y, input_len, sigma=1.5, learning_rate=0.7,
                 neighborhood_function='gaussian', topology='hexagonal', random_seed=42):
        self.x = x
        self.y = y
        np.random.seed(random_seed)
        self.weights = np.random.randn(x, y, input_len).astype(np.float32)
        self.sigma = sigma
        self.lr = learning_rate

    def random_weights_init(self, data):
        idx = np.random.choice(len(data), self.x * self.y, replace=False)
        for i, w in enumerate(idx):
            self.weights[i // self.y][i % self.y] = data[w]

    def winner(self, x):
        diff = self.weights - x
        dist = np.sum(diff ** 2, axis=2)
        idx = np.unravel_index(np.argmin(dist), dist.shape)
        return idx

    def train(self, data, num_iteration, verbose=False):
        for t in range(num_iteration):
            lr = self.lr * (1 - t / num_iteration)
            sigma = self.sigma * (1 - t / num_iteration) + 0.5
            x = data[np.random.randint(len(data))]
            bx, by = self.winner(x)
            for i in range(self.x):
                for j in range(self.y):
                    d = (i - bx) ** 2 + (j - by) ** 2
                    h = np.exp(-d / (2 * sigma ** 2))
                    self.weights[i][j] += lr * h * (x - self.weights[i][j])

# ═══════════════════════════════════════════════════════════════════════
# GLOBALS & CONFIG
# ═══════════════════════════════════════════════════════════════════════

# Debug verbosity: 0=minimal, 1=standard, 2=verbose (per-token, per-construction)
DEBUG_LEVEL = 1

# Quality-first tuning: favor syntactic exploration over speed
BEAM_WIDTH = 12
MAX_LENGTH_FACTOR = 2.0
COOCC_WEIGHT_THRESHOLD = 0.0  # Lower = more graph edges, fewer PASSTHROUGH


def _log(msg, level=1):
    if DEBUG_LEVEL >= level:
        print(msg)

# ═══════════════════════════════════════════════════════════════════════
# MOLECULAR CHEMISTRY MODEL
# ═══════════════════════════════════════════════════════════════════════
#
# Maps Akkadian morphosyntax onto molecular chemistry:
#
#   Token          → Monomer (atom with properties)
#   Sentence       → Polymer chain (backbone + side groups)
#   Affixes        → Valence electrons (fill bonding slots)
#   Case endings   → Charge (NOM=agent+, ACC=patient-, GEN=possessive~)
#   POS type       → Element class (verb=carbon backbone, noun=functional group)
#   SOV→SVO        → Conformational folding (energy minimization)
#   -ma/ša bonds   → Disulfide bridges (long-range cross-links)
#   Function words → Hydrophilic (surface/edge of sentence)
#   Content words  → Hydrophobic (cluster toward sentence core)
#
# The energy function replaces ad-hoc reordering penalties with a
# unified score: lower energy = more natural English word order.
# ═══════════════════════════════════════════════════════════════════════

# Valence table: how many bonding slots each POS type expects
# (prefix slots + suffix slots). Unsatisfied slots = instability.
VALENCE_TABLE = {
    'VERB':     4,   # person prefix + tense + root + mood/subordination
    'NOUN':     2,   # root + case ending
    'PARTICLE': 0,   # fully stable (prepositions, conjunctions)
    'UNKNOWN':  1,   # assume 1 slot (conservative)
}

# Charge by case: determines ionic attraction to other tokens
CASE_CHARGE = {
    'NOM': +1.0,    # Agent — wants to precede verb in English
    'ACC': -1.0,    # Patient — wants to follow verb in English
    'GEN': -0.5,    # Possessive — attracted to head noun
    'DAT':  0.0,    # Dative — neutral positioning
}

# Electronegativity by POS: how strongly a token attracts dependents
ELECTRONEGATIVITY = {
    'VERB':     3.5,  # High: verbs attract subjects/objects (like oxygen)
    'NOUN':     2.5,  # Medium: nouns attract modifiers
    'PARTICLE': 1.0,  # Low: function words don't attract
    'UNKNOWN':  2.0,
}

# Hydrophobicity: content words are hydrophobic (cluster to core),
# function words are hydrophilic (migrate to edges/surface)
HYDROPHOBICITY = {
    'VERB':     0.8,   # Core — backbone of the sentence
    'NOUN':     0.7,   # Core — side chains on backbone
    'PARTICLE': 0.1,   # Surface — connective tissue
    'UNKNOWN':  0.5,
}

# Bond energy constants (trained/tuned from training data)
BOND_ENERGY = {
    'covalent':    -1.0,   # Adjacent morphological agreement (strong, stabilizing)
    'ionic':       -0.6,   # Case-verb attraction (medium, directional)
    'hydrogen':    -0.3,   # Bigram co-occurrence (weak but numerous)
    'disulfide':   -2.0,   # Long-range formula bonds (very strong cross-links)
    'repulsion':   +0.5,   # Same-charge repulsion (two NOM nouns adjacent)
    'hydrophobic': -0.2,   # Content words clustering together
}


def monomer_profile(feat):
    """
    Convert a decompose_token feature dict into a molecular monomer profile.

    Noble gas rule: tokens that are bare/complete forms get charge=0,
    valence=0, stability=1.0. They don't bond or attract.
    These include: logograms, HARD_MAP particles, proper nouns,
    and any whole-word EBL/lexicon match with no prefix/suffix.
    """
    pos  = feat.get('pos', 'UNKNOWN')
    case = feat.get('case', 'NOM')
    token = feat.get('token', '')
    t_low = token.lower().strip()

    has_prefix  = bool(feat.get('prefix'))
    has_suffix  = bool(feat.get('suffix'))
    has_english = bool(feat.get('english'))

    # ── Noble gas detection ───────────────────────────────────────
    # Complete forms: no open bonds, no charge, fully inert
    is_noble = False

    # Logograms (Sumerian): fully self-contained meaning
    if feat.get('sumerian_match'):
        is_noble = True
    # HARD_MAP particles: prepositions, conjunctions — structural, not bonding
    elif t_low in HARD_MAP or feat.get('pos') == 'PARTICLE':
        is_noble = True
    # Proper nouns: names don't conjugate or decline
    elif feat.get('is_pn'):
        is_noble = True
    # Bare dictionary forms: no prefix, no suffix, but has a direct english match
    elif not has_prefix and not has_suffix and has_english:
        is_noble = True

    if is_noble:
        return {
            'pos': pos, 'case': case,
            'charge':            0.0,    # Inert — full outer shell
            'electronegativity': 0.0,    # Doesn't attract dependents
            'hydrophobicity':    0.1,    # Hydrophilic — sits where placed
            'valence':           0,
            'satisfied':         0,
            'unsatisfied':       0,
            'stability':         1.0,    # Noble gas — fully stable
            'is_verb':           feat.get('is_verb', False),
            'is_pn':             feat.get('is_pn', False),
            'is_noble':          True,
            'suffix':            feat.get('suffix', ''),
            'token':             token,
        }

    # ── Reactive token: compute charge from case ──────────────────
    # Expanded case detection (decompose_token only catches GEN/-im)
    suffix = feat.get('suffix', '')
    if suffix in ('-am', '-a') or t_low.endswith(('-am', '-a')):
        case = 'ACC'
    elif suffix in ('-iš',) or t_low.endswith('-iš'):
        case = 'DAT'
    elif suffix in ('-im', '-i') or t_low.endswith(('-im', '-i')):
        case = 'GEN'
    # Verbs are neutral — they're the backbone, not charged side-chains
    if feat.get('is_verb'):
        charge = 0.0
    else:
        charge = CASE_CHARGE.get(case, 0.0)

    # Count satisfied valence slots
    satisfied_slots = int(has_prefix) + int(has_suffix) + int(has_english)
    expected_slots  = VALENCE_TABLE.get(pos, 1)
    unsatisfied     = max(0, expected_slots - satisfied_slots)

    # Stability: 1.0 = fully bonded, 0.0 = all slots open (radical)
    stability = 1.0 - (unsatisfied / max(1, expected_slots))

    return {
        'pos':               pos,
        'case':              case,
        'charge':            charge,
        'electronegativity': ELECTRONEGATIVITY.get(pos, 2.0),
        'hydrophobicity':    HYDROPHOBICITY.get(pos, 0.5),
        'valence':           expected_slots,
        'satisfied':         satisfied_slots,
        'unsatisfied':       unsatisfied,
        'stability':         stability,
        'is_verb':           feat.get('is_verb', False),
        'is_pn':             feat.get('is_pn', False),
        'is_noble':          False,
        'suffix':            feat.get('suffix', ''),
        'token':             token,
    }


def bond_energy_pair(mono_i, mono_j, distance=1, bigram_counts=None, eng_word_i=None, eng_word_j=None):
    """
    Compute pairwise interaction energy between two monomers.
    Lower energy = more favorable adjacency in English output.

    Distance is positional gap (1 = adjacent, 2 = one word apart, etc.)
    Energy decays with distance for non-covalent interactions.
    """
    energy = 0.0

    # Noble gases don't form covalent or ionic bonds
    either_noble = mono_i.get('is_noble', False) or mono_j.get('is_noble', False)

    # 1. Covalent: morphological agreement (same clause, adjacent)
    #    Verb next to its case-marked noun = strong covalent bond
    if distance == 1 and not either_noble:
        if mono_i['is_verb'] and mono_j['pos'] == 'NOUN':
            energy += BOND_ENERGY['covalent']       # S-V or V-O bond
        elif mono_j['is_verb'] and mono_i['pos'] == 'NOUN':
            energy += BOND_ENERGY['covalent']
        # Two content words adjacent = hydrophobic clustering
        if mono_i['hydrophobicity'] > 0.5 and mono_j['hydrophobicity'] > 0.5:
            energy += BOND_ENERGY['hydrophobic']

    # 2. Ionic: charge-based attraction/repulsion (noble gases are neutral)
    if not either_noble:
        charge_product = mono_i['charge'] * mono_j['charge']
        if charge_product < 0:
            # Opposite charges attract (NOM noun near ACC noun = good, subject-object)
            energy += BOND_ENERGY['ionic'] / max(1, distance)
        elif charge_product > 0:
            # Same charges repel (two NOM nouns adjacent = bad in English)
            energy += BOND_ENERGY['repulsion'] / max(1, distance)

    # 3. Hydrogen bonds: bigram co-occurrence (weak but stabilizing)
    if bigram_counts and eng_word_i and eng_word_j and distance == 1:
        bigram_freq = bigram_counts.get((str(eng_word_i).lower(), str(eng_word_j).lower()), 0)
        if bigram_freq > 0:
            energy += BOND_ENERGY['hydrogen'] * np.log1p(bigram_freq)

    # 4. Electronegativity gradient: high-EN token pulls low-EN neighbor closer
    #    This favors verb-adjacent positions for nouns (verbs have highest EN)
    #    Noble gases (EN=0) should not participate in polar bonding
    if not either_noble:
        en_diff = abs(mono_i['electronegativity'] - mono_j['electronegativity'])
        if en_diff > 1.0 and distance == 1:
            energy += -0.1 * en_diff  # Small bonus for polar bonds

    return energy


def long_range_bonds(monomers, akk_tokens):
    """
    Detect disulfide-bridge-style long-range bonds.
    These are formula patterns that constrain folding regardless of distance.
    Returns list of (i, j, energy) tuples.
    """
    bonds = []
    toks_lower = [t.lower() for t in akk_tokens]

    # ša...ana construct chain: possessor-possessed bond
    for i, t in enumerate(toks_lower):
        if t == 'ša':
            for j in range(i+2, min(i+6, len(toks_lower))):
                if toks_lower[j] == 'ana' or toks_lower[j] == 'a-na':
                    bonds.append((i, j, BOND_ENERGY['disulfide']))
                    break

    # -ma clause boundary: links pre-ma content to post-ma content
    for i, mono in enumerate(monomers):
        if mono['suffix'] == '-ma' and i + 1 < len(monomers):
            bonds.append((i, i+1, BOND_ENERGY['disulfide'] * 0.5))

    # šu-ma...ú-ṣa-áb conditional-penalty bridge
    shuma_idx = None
    for i, t in enumerate(toks_lower):
        if t in ('šu-ma', 'su-ma'):
            shuma_idx = i
        if shuma_idx is not None and t in ('ú-ṣa-áb', 'u-sa-ab'):
            bonds.append((shuma_idx, i, BOND_ENERGY['disulfide']))
            break

    return bonds


def chain_energy(monomers, eng_words, bigram_counts=None, lr_bonds=None):
    """
    Total energy of the molecular chain (sentence) in its current conformation.
    Sum of all pairwise interactions + long-range bonds + stability penalties.
    Lower = better English word order.
    """
    n = len(monomers)
    total = 0.0

    # Pairwise interactions (up to distance 3 — beyond that, negligible)
    for i in range(n):
        for j in range(i+1, min(i+4, n)):
            ew_i = eng_words[i] if i < len(eng_words) else None
            ew_j = eng_words[j] if j < len(eng_words) else None
            total += bond_energy_pair(
                monomers[i], monomers[j],
                distance=j-i,
                bigram_counts=bigram_counts,
                eng_word_i=ew_i, eng_word_j=ew_j
            )

    # Long-range bonds (disulfide bridges)
    if lr_bonds:
        for i, j, e in lr_bonds:
            if i < n and j < n:
                total += e

    # Stability penalty: unstable monomers (unsatisfied valence) raise energy
    for mono in monomers:
        total += 0.3 * mono['unsatisfied']  # Each open slot costs +0.3

    # Hydrophobic effect: content words at sentence edges are penalized
    if n >= 3:
        # First and last positions prefer low hydrophobicity (function words)
        total += 0.2 * monomers[0]['hydrophobicity']   # Penalty for content at start
        total += 0.2 * monomers[-1]['hydrophobicity']   # Penalty for content at end

    return total


def molecular_reorder(segment_entries, bigram_counts=None):
    """
    Energy-minimized reordering. Instead of mechanical SOV→SVO verb-flip,
    tries permutations within each clause and picks the lowest-energy fold.

    For short clauses (≤8 tokens), does exhaustive search.
    For longer clauses, uses the mechanical flip as seed then tries
    local swaps (simulated annealing–lite).
    """
    from itertools import permutations

    if len(segment_entries) < 2:
        return segment_entries

    # Split into clauses at boundaries (same logic as before)
    clauses = []
    current_clause = []
    for entry in segment_entries:
        cands, feats, route, span_toks = entry
        current_clause.append(entry)
        tok = span_toks[0].lower() if span_toks else ''
        suffix = feats[0].get('suffix', '') if feats else ''
        if suffix in _CLAUSE_BOUNDARY_SUFFIXES or tok in _CLAUSE_BOUNDARY_TOKENS:
            clauses.append(current_clause)
            current_clause = []
    if current_clause:
        clauses.append(current_clause)

    reordered = []
    for clause in clauses:
        if len(clause) <= 1:
            reordered.extend(clause)
            continue

        # Build monomers for this clause
        clause_monomers = []
        clause_eng_words = []
        clause_akk_tokens = []
        for cands, feats, route, span_toks in clause:
            feat = feats[0] if feats else {}
            clause_monomers.append(monomer_profile(feat))
            clause_eng_words.append(str(cands[0][0]) if cands else '<gap>')
            clause_akk_tokens.append(span_toks[0] if span_toks else '')

        lr_bonds = long_range_bonds(clause_monomers, clause_akk_tokens)

        if len(clause) <= 6:
            # Exhaustive: try all permutations, pick lowest energy
            best_perm = None
            best_energy = float('inf')
            indices = list(range(len(clause)))
            # Limit permutations for safety (6! = 720, manageable)
            for perm in permutations(indices):
                perm_monomers = [clause_monomers[p] for p in perm]
                perm_words    = [clause_eng_words[p] for p in perm]
                e = chain_energy(perm_monomers, perm_words, bigram_counts, lr_bonds)
                if e < best_energy:
                    best_energy = e
                    best_perm = perm
            reordered.extend([clause[p] for p in best_perm])
            _log(f"[CHEM] Clause ({len(clause)} tokens): exhaustive fold, energy={best_energy:.3f}", level=2)
        else:
            # Longer clause: start with mechanical SVO flip, then local swaps
            base_order = _reorder_clause_svo(clause)
            base_monomers = []
            base_words = []
            for cands, feats, route, span_toks in base_order:
                feat = feats[0] if feats else {}
                base_monomers.append(monomer_profile(feat))
                base_words.append(str(cands[0][0]) if cands else '<gap>')

            best_energy = chain_energy(base_monomers, base_words, bigram_counts, lr_bonds)
            best_order  = list(base_order)

            # Try adjacent swaps (bubble-sort style annealing)
            improved = True
            max_rounds = 3
            round_num = 0
            while improved and round_num < max_rounds:
                improved = False
                round_num += 1
                for i in range(len(best_order) - 1):
                    trial = list(best_order)
                    trial[i], trial[i+1] = trial[i+1], trial[i]
                    trial_monomers = []
                    trial_words = []
                    for cands, feats, route, span_toks in trial:
                        feat = feats[0] if feats else {}
                        trial_monomers.append(monomer_profile(feat))
                        trial_words.append(str(cands[0][0]) if cands else '<gap>')
                    e = chain_energy(trial_monomers, trial_words, bigram_counts, lr_bonds)
                    if e < best_energy:
                        best_energy = e
                        best_order = trial
                        improved = True
            reordered.extend(best_order)
            _log(f"[CHEM] Clause ({len(clause)} tokens): annealed fold, energy={best_energy:.3f}", level=2)

    return reordered


def valence_beam_penalty(monomers):
    """
    Beam search penalty based on molecular stability.
    Sequences with many unsatisfied valence slots are penalized.
    Returns a negative score (penalty).
    """
    if not monomers:
        return 0.0
    total_instability = sum(m['unsatisfied'] for m in monomers)
    avg_stability = sum(m['stability'] for m in monomers) / len(monomers)
    # Reward high average stability, penalize total instability
    return 0.2 * avg_stability - 0.15 * total_instability


def train_bond_energies(train_pairs, akk_token_to_cluster):
    """
    Learn bond energy parameters from training data.
    For each training pair, we know the correct English order.
    Extract which POS adjacencies appear most often in correct translations
    and adjust bond energies to favor those patterns.
    """
    # Count POS-pair adjacencies in correct English translations
    pos_pair_counts = Counter()
    case_pair_counts = Counter()
    total_pairs = 0

    for akk_tokens, eng_sent in train_pairs:
        eng_words = eng_sent.lower().split()
        if len(eng_words) < 2:
            continue
        # We don't have POS tags for English, but we can infer patterns:
        # Track which Akkadian POS types appear adjacent in aligned output
        akk_feats = [decompose_token(tok) for tok in akk_tokens]
        akk_monomers = [monomer_profile(f) for f in akk_feats]
        for i in range(len(akk_monomers) - 1):
            pair = (akk_monomers[i]['pos'], akk_monomers[i+1]['pos'])
            pos_pair_counts[pair] += 1
            case_pair = (akk_monomers[i]['case'], akk_monomers[i+1]['case'])
            case_pair_counts[case_pair] += 1
            total_pairs += 1

    if total_pairs == 0:
        return

    # Adjust covalent bond energy: VERB-NOUN pairs are strongest
    vn_freq = (pos_pair_counts.get(('VERB', 'NOUN'), 0) +
               pos_pair_counts.get(('NOUN', 'VERB'), 0)) / max(1, total_pairs)
    nn_freq = pos_pair_counts.get(('NOUN', 'NOUN'), 0) / max(1, total_pairs)

    # Scale covalent bond by how common V-N adjacency is
    BOND_ENERGY['covalent'] = -1.0 * (1 + vn_freq * 2)

    # Scale repulsion by how uncommon same-case adjacency is
    nom_nom = case_pair_counts.get(('NOM', 'NOM'), 0) / max(1, total_pairs)
    BOND_ENERGY['repulsion'] = 0.5 * (1 + nom_nom * 3)

    _log(f"[CHEM] Trained bond energies from {total_pairs} pairs: "
         f"covalent={BOND_ENERGY['covalent']:.3f}, repulsion={BOND_ENERGY['repulsion']:.3f}")


# ═══════════════════════════════════════════════════════════════════════
# SOM CLUSTERING
# ═══════════════════════════════════════════════════════════════════════

def build_som_clusters(tokens, n_cols=4, n_rows=48, n_iter=3000, random_seed=42):
    """
    Train a SOM on token TF-IDF vectors.
    Returns: dict mapping token -> flat cluster ID (0 to n_cols*n_rows - 1)

    Flat ID = row * n_cols + col
    This preserves drop-in compatibility with KMeans cluster integers.
    Topology is hexagonal (honeycomb).
    """
    vectorizer = TfidfVectorizer()
    tfidf = vectorizer.fit_transform(tokens).toarray().astype(np.float32)
    n_features = tfidf.shape[1]

    som = MiniSom(
        n_cols, n_rows, n_features,
        sigma=1.5, learning_rate=0.7,
        neighborhood_function='gaussian',
        topology='hexagonal',
        random_seed=random_seed
    )
    som.random_weights_init(tfidf)
    som.train(tfidf, num_iteration=n_iter, verbose=False)

    token_to_cluster = {}
    for i, tok in enumerate(tokens):
        col, row = som.winner(tfidf[i])
        flat_id = row * n_cols + col
        token_to_cluster[tok] = flat_id

    n_clusters = n_cols * n_rows
    _log(f"[SOM] Grid: {n_cols}x{n_rows} = {n_clusters} cells | "
         f"tokens: {len(tokens)} | features: {n_features}")
    occupied = len(set(token_to_cluster.values()))
    _log(f"[SOM] Occupied cells: {occupied}/{n_clusters} "
         f"({occupied/n_clusters*100:.1f}%)")
    return token_to_cluster, som, vectorizer


def build_all_clusters(all_akk_tokens, all_eng_words):
    """
    Builds akk_token_to_cluster and eng_word_to_cluster using SOM.
    Same interface as KMeans version — returns two dicts.
    n_clusters stays 56 (7x8) to match original 55 as closely as possible.
    """
    _log(f"\n[CLUSTER] Akkadian: {len(all_akk_tokens)} unique tokens")
    akk_token_to_cluster, akk_som, _ = build_som_clusters(
        all_akk_tokens, n_cols=7, n_rows=8, n_iter=3000
    )
    akk_cluster_dist = Counter(akk_token_to_cluster.values())
    _log(f"[CLUSTER] Akk SOM cluster distribution (top 5): "
         f"{akk_cluster_dist.most_common(5)}")

    _log(f"[CLUSTER] English: {len(all_eng_words)} unique words")
    eng_word_to_cluster, eng_som, _ = build_som_clusters(
        all_eng_words, n_cols=7, n_rows=8, n_iter=3000
    )
    return akk_token_to_cluster, eng_word_to_cluster


# ═══════════════════════════════════════════════════════════════════════
# DATA LOADING & LEXICON SETUP
# ═══════════════════════════════════════════════════════════════════════

def _find_data_dir():
    candidates = [
        Path("/kaggle/input/deep-past-initiative-machine-translation"),
        Path("/kaggle/input/deep-past-challenge-translate-akk"),
        Path("."),
    ]
    for p in candidates:
        if (p / "train.csv").exists() and (p / "eBL_Dictionary.csv").exists():
            return p
    if os.path.exists("/kaggle/input"):
        for root, _, files in os.walk("/kaggle/input"):
            if "train.csv" in files and "eBL_Dictionary.csv" in files:
                return Path(root)
    return Path("/kaggle/input/deep-past-initiative-machine-translation")

DATA_DIR = _find_data_dir()
_log(f"\n{'='*70}")
_log("SANALVARO - Akkadian Translation Pipeline")
_log(f"{'='*70}")
_log(f"[CONFIG] DATA_DIR = {DATA_DIR}")
_log(f"[CONFIG] DEBUG_LEVEL = {DEBUG_LEVEL}")
_log(f"[CONFIG] train.csv exists: {(DATA_DIR / 'train.csv').exists()}")
_log(f"[CONFIG] test.csv exists: {(DATA_DIR / 'test.csv').exists()}")
_log(f"[CONFIG] eBL_Dictionary.csv exists: {(DATA_DIR / 'eBL_Dictionary.csv').exists()}")

# ── eBL Dictionary ────────────────────────────────────────────────────
ebl_df = pd.read_csv(DATA_DIR / "eBL_Dictionary.csv")

def _extract_meaning(defn):
    if pd.isna(defn):
        return None
    m = re.search(r'"([^"]+)"', str(defn))
    if not m:
        return None
    meaning = m.group(1).strip()
    if meaning.startswith("~") or len(meaning) < 2:
        return None
    if meaning.startswith("to "):
        meaning = meaning[3:]
    meaning = meaning.split(",")[0].split(";")[0].strip()
    meaning = re.sub(r'[():]', '', meaning).strip()
    if len(meaning) > 30 or meaning.count(" ") > 3:
        return None
    return meaning

_LEX_SUB = str.maketrans('\u2080\u2081\u2082\u2083\u2084\u2085\u2086\u2087\u2088\u2089', '0123456789')
def _norm_key(s):
    """Normalize a lexicon key to match cleaned test tokens."""
    return (str(s).strip().lower()
            .replace('\u1e2b', 'h').replace('\u1e2a', 'h')
            .replace('í', 'i').replace('ú', 'u').replace('á', 'a').replace('é', 'e')
            .replace('\u201e', '').replace('\u201c', '').replace('"', '')
            .translate(_LEX_SUB))

EBL_LEXICON = {}
for _, r in ebl_df.iterrows():
    w, d = r.get("word"), r.get("definition")
    if pd.notna(w) and pd.notna(d):
        meaning = _extract_meaning(d)
        if meaning:
            key = _norm_key(w)
            if key and key not in EBL_LEXICON:
                EBL_LEXICON[key] = meaning
_log(f"[EBL] Loaded {len(EBL_LEXICON)} entries from eBL_Dictionary.csv")

# ── OA Quality Patch: force-inject full-word mappings ─────────────────
OA_QUALITY_PATCH = {
    'mup-pì-im': 'the word', 'mup-pì-ni': 'our word', 'mup-pu-um': 'the statement',
    'muppim': 'the word', 'muppini': 'our word', 'muppum': 'the statement',
    'kà-ru-um': 'the colony', 'kà-ni-ia': 'Kanesh', 'karaum': 'the colony', 'kania': 'Kanesh',
    'wa-bar-ra-tim': 'the trading stations', 'wabarratim': 'the trading stations',
    'qí-bi-ma': 'speak', 'qibima': 'speak', 'um-ma': 'thus says', 'umma': 'thus says',
    'u4-mì-im': 'day', 'umim': 'day', 'a-nim': 'this', 'anim': 'this',
    'ma-ma-an': 'anybody', 'maman': 'anybody', 'né-mì-lim': 'profit', 'nemilim': 'profit',
    'ia-tí': 'me', 'iati': 'me', 'a-lim(ki)': 'City', 'kà-ar': 'trading post',
    'kà-ar-ma': 'and to the trading post', 'u-mì-im': 'day',
    'me-e-er': 'copy', 'me-er': 'copy', 'mehru': 'copy',
    'DAM.GÀR-ru-tim': 'the merchants', 'a-wi-lim': 'the man', 'awilim': 'the man',
    'lu-up-ta-nim': 'be recorded', 'luptanim': 'be recorded',
    'aí-ip-ri-ni': 'provide', 'aiprini': 'provide', 'a-pu-tum': 'urgent',
    'i-hi-id-ma': 'pay attention', 'ú-lá': 'not', 'ni-bi„-it': 'witnesses',
    'tí-ir-ta-kà': 'your report', 'ú-kà-lim': 'palace officials',
    'lá ta-ba-ta-aq': 'do not sell at a loss', 'a-dí-ni': 'still',
    'pí-qí-id-ma': 'entrust it', 'lu-up-ta-nim-ma': 'and be recorded',
    'i-se-er': 'against', 'ú-ṣa-áb': 'he will add',
    'i-ša-qal': 'he will pay', 'ša-na-at': 'year', 'ha-ar-pe': 'harvest time',
}
EBL_LEXICON.update(OA_QUALITY_PATCH)
_log(f"[EBL] OA_QUALITY_PATCH: +{len(OA_QUALITY_PATCH)} entries")
if DEBUG_LEVEL >= 1 and EBL_LEXICON:
    sample = list(EBL_LEXICON.items())[:5]
    _log(f"[EBL] Sample: {sample}")

# ── Hard-map prepositions ─────────────────────────────────────────────
HARD_MAP = {
    'a-na': 'to', 'ana': 'to', 'ki-ma': 'just as', 'i-na': 'in', 'ina': 'in',
    'šu-ma': 'if', 'u-lá': 'not', 'mi-ma': 'anything', 'a-pu-tum': 'urgent',
    'ša': 'of', 'u': 'and', 'ú': 'and', 'la': 'not', 'lá': 'not',
    'lu': 'let', 'ma-lá': 'as much as',
}

# ── CAD Dictionary ────────────────────────────────────────────────────
cad_df = pd.read_csv('/kaggle/input/datasets/alvarochaveste/akkylexyfromdic/akkadian_lexiconfromdick.csv')
cad_df2 = pd.read_csv('/kaggle/input/datasets/alvarochaveste/akkylexyfromdic/akkadian_lexiconfromdick1-74.csv')
cad_df = pd.concat([cad_df, cad_df2], ignore_index=True).drop_duplicates(subset='headword')
cad_df = cad_df[['headword', 'grammar', 'definition']]
CAD_DICT = dict(zip(cad_df['headword'].map(_norm_key), cad_df['definition']))
_log(f"[CAD] Loaded {len(CAD_DICT)} entries from akkadian_lexiconfromdick.csv")


cad_clean = pd.read_csv('/kaggle/input/datasets/alvarochaveste/akkadian-lexicon-clean/akkadian_lexicon_clean.csv')
cad_clean = cad_clean[cad_clean['gloss'].notna() & (cad_clean['gloss'].str.len() > 2)]
CAD_DICT.update(dict(zip(
    cad_clean['headword'].map(_norm_key),
    cad_clean['gloss']
)))
_log(f"[CAD] Lexicon loaded: {len(CAD_DICT)} entries")

# ── Onomasticon ───────────────────────────────────────────────────────
ono_df = pd.read_csv('/kaggle/input/datasets/deeppast/old-assyrian-grammars-and-other-resources/onomasticon.csv')
_ONO_SUB = str.maketrans('\u2080\u2081\u2082\u2083\u2084\u2085\u2086\u2087\u2088\u2089', '0123456789')
def _clean_ono(s):
    return (s.strip()
             .replace('<big_gap>', '<gap>')
             .translate(_ONO_SUB)
             .replace('\u1e2a', 'H').replace('\u1e2b', 'h'))
ONOMASTICON = set(ono_df.iloc[:, 0].dropna().map(_clean_ono))
ONO_SPELLING_MAP = {}
for _, row in ono_df.iterrows():
    canonical = _clean_ono(str(row['Name'])) if pd.notna(row['Name']) else None
    if not canonical:
        continue
    ONO_SPELLING_MAP[canonical.lower()] = canonical
    spellings = str(row['Spellings_semicolon_separated']) if pd.notna(row['Spellings_semicolon_separated']) else ''
    for sp in spellings.split(';'):
        sp = _clean_ono(sp)
        if sp:
            ONO_SPELLING_MAP[sp.lower()] = canonical
_log(f"[ONOMASTICON] {len(ONO_SPELLING_MAP)} spelling variants -> {len(ONOMASTICON)} canonical names")

# ── Stem-to-Infinitive mapper ────────────────────────────────────────
INFLECTED_TO_ROOT = {
    'i-dí-in': 'nadānum', 'i-dí-nu': 'nadānum', 'ta-da-na-at': 'nadānum',
    'iddin': 'nadānum', 'iddinu': 'nadānum', 'tadanat': 'nadānum',
    'ta-áa-me-a-ni': 'šemûm', 'ta-a-me-a-ni': 'šemûm', 'šemûni': 'šemûm',
    'i-lá-qé': 'leqûm', 'ilaqe': 'leqûm', 'ilqe': 'leqûm',
    'i-li-kam': 'alākum', 'ilikum': 'alākum', 'ilikam': 'alākum',
    'aé-bi-lá': 'wabālum', 'a-bi-la': 'wabālum', 'abilam': 'wabālum',
    'id-din': 'nadānum', 'id-di-nu': 'nadānum', 'šé-bi-lam': 'wabālum',
    'šé-bi-lá-nim': 'wabālum', 'té-ra-at': 'târum',
    'ta-áš-pu-ra-ni': 'šapārum', 'li-li-kam': 'alākum', 'id-na-a': 'nadānum',
    'ku-un-kà-ma': 'kanākum', 'té-pu-šu': 'epēšum',
    'i-ba-ší': 'bašûm', 'ta-lá-qé': 'leqûm', 'ú-ta-ar': 'târum',
    'i-mu-a': 'mâum', 'ta-ṣú-ra-am': 'naṣārum', 'ú-ša-qí-lu': 'šaqālum',
    'i-pá-ṭá-ar': 'paṭārum', 'i-za-bi₄-lam': 'wabālum',
    'i-ṣé-er': 'ṣērum', 'i-šu': 'išûm', 'i-ša-qal': 'šaqālum', 'ú-ṣa-áb': 'waṣābum',
}

# ── Quality lexicon: infinitive -> English ────────────────────────────
QUALITY_LEXICON = {
    'nadānum': 'give', 'šemûm': 'hear', 'leqûm': 'take', 'alākum': 'come',
    'wabālum': 'send', 'amūtum': 'tin', 'nadātum': 'deposit', 'qabûm': 'say',
    'eperum': 'provide', 'šapārum': 'write', 'kanākum': 'seal', 'epēšum': 'do',
    'bašûm': 'exist', 'târum': 'return', 'mâum': 'refuse', 'naṣārum': 'guard',
    'paṭārum': 'release', 'lapātum': 'record', 'ṣērum': 'against',
    'išûm': 'owes', 'waṣābum': 'add interest',
}

# ── Kültepe particles ─────────────────────────────────────────────────
KULTEPE_PARTICLES = {
    'lu': 'let', 'la': 'not', 'ma-la': 'as much as', 'a-ma-kam': 'there',
}

# ── Sumerian logograms ────────────────────────────────────────────────
SUMERIAN_LOGO = {
    'KÙ.AN': 'tin', 'KÙ.BABBAR': 'silver', 'É.GAL': 'palace',
    'LUGAL': 'king', 'DINGIR': 'god', 'KUR': 'mountain', 'ŠU': 'hand',
    'AN.NA': 'heaven', 'LAM+KUR': 'hostile', 'SAG.GIŠ.RA': 'to smite',
}


# ═══════════════════════════════════════════════════════════════════════
# TOKEN ANALYSIS FUNCTIONS
# ═══════════════════════════════════════════════════════════════════════

def _normalize_for_fuzzy(s):
    s = s.replace('„', '').replace('+', '').replace('…', '')
    s = unicodedata.normalize('NFKD', s)
    return s.lower().strip()


def _fuzzy_lookup(token, min_len=4, cutoff=0.80):
    try:
        norm_tok = _normalize_for_fuzzy(token)
        if len(norm_tok) < min_len:
            return None, None
        all_known = {
            _normalize_for_fuzzy(k): (k, v)
            for k, v in {**EBL_LEXICON, **CAD_DICT}.items()
            if v
        }
        matches = get_close_matches(norm_tok, all_known.keys(), n=1, cutoff=cutoff)
        if matches:
            orig_key, eng = all_known[matches[0]]
            eng = str(eng)
            if not eng or eng.startswith('(') or len(eng) < 3 or eng.startswith('~'):
                return None, None
            return eng, orig_key
        return None, None
    except Exception:
        return None, None


def _token_to_root(token):
    """Resolve token to dictionary root (for graph nodes)."""
    t = str(token).lower().strip()
    t_norm = t.replace('í', 'i').replace('ú', 'u').replace('á', 'a').replace('é', 'e')
    return INFLECTED_TO_ROOT.get(t) or INFLECTED_TO_ROOT.get(t_norm) or extract_semitic_root(token)


def extract_semitic_root(token):
    """
    Handles common Akkadian phonological shifts to find the dictionary root.
    CAD/EBL dictionaries list words by infinitive (e.g., nadanum instead of iddin).
    """
    t = token.lower()

    # 1. Handle common 'n' assimilation
    if re.match(r'^id-d', t) or re.match(r'^idd', t):
        return 'nadānum'
    if re.match(r'^iq-q', t) or re.match(r'^iqq', t):
        return 'naqûm'

    # 2. Handle 't' infixes (G-Perfect / Gt-Stems)
    t_infix = re.search(r'^([aeiu][b-df-hj-np-rt-vz])t([b-df-hj-np-rt-vz])', t)
    if t_infix:
        t = t_infix.group(1) + t_infix.group(2) + t[t_infix.end():]

    # 3. Clean up vocalic endings (Case/Mood)
    t = re.sub(r'[uia]m?$', '', t)

    # Guard: don't return over-stripped roots shorter than 3 chars
    if len(t.replace('-', '')) < 3:
        return token
    return t


def decompose_token(token):
    """
    Quality-focused morphological decomposition. Lookup order:
    HARD_MAP > Kültepe > Logogram > QUALITY_LEXICON > EBL > Graph (never guess stop-words).
    """
    token = token.replace('„', '"').replace('+', '').replace('"', '').replace("'", '').replace('…', '').strip()
    t_low = token.lower().strip()
    t_norm = t_low.replace('í', 'i').replace('ú', 'u').replace('á', 'a').replace('é', 'e').replace('ḫ', 'h').replace('ḥ', 'h')

    # 1. HARD_MAP: prepositions/particles
    if t_low in HARD_MAP or t_norm in HARD_MAP:
        eng = HARD_MAP.get(t_low) or HARD_MAP.get(t_norm)
        return {
            'token': token, 'root': t_low, 'sem_root': t_low, 'prefix': '', 'suffix': '',
            'norm': t_low, 'semantic': 'UNKNOWN', 'pos': 'PARTICLE', 'is_pn': False,
            'sumerian_match': None, 'cad_root': t_low, 'cad_suffix': '', 'english': eng,
            'is_verb': False, 'case': 'NOM',
        }

    # 2. Kültepe (Old Assyrian) particles
    if t_low in KULTEPE_PARTICLES:
        return {
            'token': token, 'root': t_low, 'sem_root': t_low, 'prefix': '', 'suffix': '',
            'norm': t_low, 'semantic': 'UNKNOWN', 'pos': 'PARTICLE', 'is_pn': False,
            'sumerian_match': None, 'cad_root': t_low, 'cad_suffix': '',
            'english': KULTEPE_PARTICLES[t_low], 'is_verb': False, 'case': 'NOM',
        }

    # 3. Logogram overrides (Old Assyrian Kültepe-specific)
    token_upper = token.upper()
    if 'KÙ.AN' in token_upper:
        english_override = 'tin'
    elif 'KÙ.BABBAR' in token_upper:
        english_override = 'silver'
    elif 'É.GAL' in token_upper:
        english_override = 'palace'
    else:
        english_override = SUMERIAN_LOGO.get(token_upper, None)

    # Prefix/suffix patterns
    prefix_pattern = r'^(u-|la-|ša-|ana-|ina-|i-|ma-|v-|li-|lu-)'
    suffix_pattern = r'(-ma|-um|-im|-am|-šu|-ša|-ka|-ki|-ni|-šunu|-šina|-lim|-kam)$'
    p_match = re.match(prefix_pattern, token)
    s_match = re.search(suffix_pattern, token)
    prefix = p_match.group(0) if p_match else ''
    suffix = s_match.group(0) if s_match else ''
    base_root = token[len(prefix):len(token) - len(suffix)] if (prefix or suffix) else token

    # 4. Strong dictionary lookup: inflected -> infinitive -> QUALITY_LEXICON
    root = (INFLECTED_TO_ROOT.get(t_low) or INFLECTED_TO_ROOT.get(t_norm) or
            INFLECTED_TO_ROOT.get(token, None))
    if root is None:
        root = extract_semitic_root(base_root) if (prefix or suffix) else extract_semitic_root(token)
    semantic_root = root if root else extract_semitic_root(base_root)

    # 5. Verb markers (for SOV->SVO reordering)
    is_verb = (
        (t_low.startswith(('i-', 'u-', 'ta-', 'a-', 'lu-')) or
         prefix in ['i-', 'u-', 'ta-', 'a-', 'lu-'])
        and t_low not in HARD_MAP
        and t_low not in KULTEPE_PARTICLES
    )
    pos = 'VERB' if is_verb else ('NOUN' if suffix in ['-um', '-im', '-am'] else 'UNKNOWN')
    case = 'GEN' if (t_low.endswith(('-im', '-i')) or suffix in ['-im', '-i']) else 'NOM'

    # 6. Proper noun detection
    is_pn = (token in ONOMASTICON or
             (any(c.isupper() for c in token) and '.' not in token))

    # 7. English: logogram override > QUALITY_LEXICON > EBL
    english = english_override
    if not english and semantic_root:
        english = QUALITY_LEXICON.get(semantic_root, None)
    if not english:
        for candidate in [t_low, t_norm, semantic_root, base_root]:
            if not candidate:
                continue
            if len(candidate.replace('-', '')) < 4:
                continue
            if candidate in EBL_LEXICON:
                english = EBL_LEXICON[candidate]
                break

    # Conjunction -ma
    if suffix == '-ma' and english and not english.startswith('and '):
        english = "and " + english
    elif suffix == '-ma' and not english:
        english = "and " + base_root

    cad_root = CAD_DICT.get(token, semantic_root)
    semantic = 'COMMERCE' if any(k in base_root.lower() for k in ['kasp', 'tamkar']) else 'UNKNOWN'

    return {
        'token': token, 'root': base_root, 'sem_root': semantic_root,
        'prefix': prefix, 'suffix': suffix, 'norm': t_low, 'semantic': semantic,
        'pos': pos, 'is_pn': is_pn, 'sumerian_match': english_override,
        'cad_root': cad_root, 'cad_suffix': suffix, 'english': english,
        'is_verb': is_verb, 'case': case,
    }


# ═══════════════════════════════════════════════════════════════════════
# CONSTRUCTION IDENTIFICATION & TEMPLATES
# ═══════════════════════════════════════════════════════════════════════

def identify_construction_types(features, context, idx=0):
    constructions = []
    # Kültepe: ma-la sequence -> "as much as"
    if idx + 1 < len(context) and context[idx].lower() == 'ma' and context[idx + 1].lower() == 'la':
        constructions.append({
            'type': 'kultepe', 'pattern': 'MA_LA',
            'slots': {}, 'start_idx': idx, 'end_idx': idx + 2,
        })
    # Morphological
    if features['suffix'] == '-ma':
        full_tok = features['token'].lower().strip()
        if full_tok not in EBL_LEXICON and full_tok not in HARD_MAP:
            constructions.append({
                'type': 'morphological', 'pattern': '-ma',
                'slots': {'root': features['root'], 'sem_root': features.get('sem_root', features['root'])}
            })
    # Lexical
    if 'kasp' in features['root']:
        constructions.append({'type': 'lexical', 'pattern': 'kaspum', 'slots': {'root': features['root']}})
    # Frame
    if features['semantic'] == 'COMMERCE':
        constructions.append({'type': 'frame', 'pattern': 'COMMERCE',
                              'slots': {'item': 'silver', 'recipient': 'merchant'}})
    # Syntactic: construct chain (ša ... ana ...)
    for i in range(len(context) - 3):
        if context[i].lower() == 'ša' and context[i+2].lower() == 'ana':
            possessed_root = decompose_token(context[i+1])['root']
            possessor_root = decompose_token(context[i+3])['root']
            constructions.append({
                'type': 'syntactic', 'pattern': 'CONSTRUCT_CHAIN',
                'slots': {'possessed': possessed_root, 'possessor': possessor_root},
                'start_idx': i, 'end_idx': i+4
            })
    return constructions


construction_templates = {
    'kultepe_MA_LA': "as much as",
    'morphological_-ma': "and {root}",
    'lexical_kaspum': "{root}",
    'frame_COMMERCE': "the {item} to the {recipient}",
    'syntactic_CONSTRUCT_CHAIN': "{possessor}'s {possessed}",
}


# ═══════════════════════════════════════════════════════════════════════
# SCORING & VALIDATION
# ═══════════════════════════════════════════════════════════════════════

def compute_composite_similarity(akk_feat, eng_words, vectorizer, length_ratio=1.0, semantic_density=0.5):
    akk_str = ' '.join([str(v) for v in akk_feat.values() if isinstance(v, str)])
    eng_str = ' '.join(eng_words)
    vectors = vectorizer.transform([akk_str, eng_str])
    cos_sim = cosine_similarity(vectors[0], vectors[1])[0][0]
    akk_set = set(akk_feat.values())
    eng_set = set(eng_words)
    jac = len(akk_set.intersection(eng_set)) / len(akk_set.union(eng_set)) if akk_set.union(eng_set) else 0
    return (
        cos_sim * 0.25 +
        jac * 0.2 +
        (1 / length_ratio if length_ratio > 0 else 0) * 0.15 +
        semantic_density * 0.15
    )


def get_top_eng_words(source, mapping, top_n=3):
    if source in mapping:
        return [(w, p) for w, p in mapping[source].most_common(top_n)]
    return []


def score_partial_sequence(partial_eng, akk_feats, bigram_counts, vectorizer):
    """Holistic score: composite + bigram log-prob + length penalty."""
    if len(partial_eng) < 2:
        return 0.0
    partial_lower = [str(w).lower() for w in partial_eng]
    composite = compute_composite_similarity(
        akk_feats[0], partial_lower, vectorizer,
        length_ratio=max(0.1, len(partial_eng) / max(1, len(akk_feats)))
    )
    bigram_score = sum(
        np.log(bigram_counts.get((partial_lower[i], partial_lower[i + 1]), 1e-6))
        for i in range(len(partial_lower) - 1)
    )
    length_penalty = -0.1 * abs(len(partial_eng) - len(akk_feats))
    return composite + bigram_score / max(1, len(partial_eng)) + length_penalty


def forward_backward_validate(eng_seq, akk_tokens, root_mapping):
    """Back-prediction: map English words to Akk roots, check overlap with original."""
    back_roots = set()
    for word in eng_seq.lower().split():
        for root, counter in root_mapping.items():
            if word in counter:
                back_roots.add(root)
    orig_roots = set(decompose_token(tok)['root'] for tok in akk_tokens)
    overlap = len(back_roots.intersection(orig_roots)) / len(orig_roots) if orig_roots else 0
    return overlap > 0.5


# ═══════════════════════════════════════════════════════════════════════
# SOV -> SVO REORDERING
# ═══════════════════════════════════════════════════════════════════════

def _reorder_clause_svo(clause_entries):
    """SOV -> SVO reorder for a single clause. Handles 4-tuple entries."""
    if len(clause_entries) < 2:
        return clause_entries
    first_noun_idx = None
    verb_idx       = None
    for i, (_, feats, _, _) in enumerate(clause_entries):
        pos     = feats[0].get('pos', 'UNKNOWN') if feats else 'UNKNOWN'
        is_verb = feats[0].get('is_verb', False) if feats else False
        if pos == 'NOUN' and first_noun_idx is None:
            first_noun_idx = i
        if (pos == 'VERB' or is_verb) and verb_idx is None:
            verb_idx = i
    if first_noun_idx is None or verb_idx is None or verb_idx <= first_noun_idx:
        return clause_entries
    subject_part = clause_entries[:first_noun_idx + 1]
    object_part  = clause_entries[first_noun_idx + 1:verb_idx]
    verb_part    = [clause_entries[verb_idx]]
    rest         = clause_entries[verb_idx + 1:]
    return subject_part + verb_part + object_part + rest


_CLAUSE_BOUNDARY_SUFFIXES = {'-ma'}
_CLAUSE_BOUNDARY_TOKENS   = {'ša'}

def _reorder_segments_for_svo(segment_entries):
    """
    Clause-aware SOV -> SVO reorder.
    Splits at clause boundaries (-ma, ša, šu-ma), reorders each independently.
    """
    if len(segment_entries) < 2:
        return segment_entries
    clauses = []
    current_clause = []
    for entry in segment_entries:
        cands, feats, route, span_toks = entry
        current_clause.append(entry)
        tok = span_toks[0].lower() if span_toks else ''
        suffix = feats[0].get('suffix', '') if feats else ''
        if suffix in _CLAUSE_BOUNDARY_SUFFIXES or tok in _CLAUSE_BOUNDARY_TOKENS:
            clauses.append(current_clause)
            current_clause = []
    if current_clause:
        clauses.append(current_clause)
    reordered = []
    for clause in clauses:
        reordered.extend(_reorder_clause_svo(clause))
    return reordered


def _get_graph_path(G, source, target):
    """Return shortest path from source to target as string, or None."""
    try:
        path = nx.dijkstra_path(G, source, target,
                                weight=lambda u, v, d: -np.log(d.get('weight', 1e-6) + 1e-6))
        return " -> ".join(str(n) for n in path)
    except (nx.NetworkXNoPath, nx.NodeNotFound):
        return None


# ═══════════════════════════════════════════════════════════════════════
# FORMULA DETECTORS
# Pre-pass: check for high-confidence formulaic patterns before
# compose_sequence runs. Priority: letter opening > debt > payment > penalty
# ═══════════════════════════════════════════════════════════════════════

def detect_formula(akk_tokens):
    """
    Check token list for known Old Assyrian commercial formulas.
    Returns (formula_type, result_dict) or (None, None).
    """
    toks = [t.lower().strip() for t in akk_tokens]
    toks_orig = akk_tokens

    # ── 1. LETTER OPENING ─────────────────────────────────────────
    if 'qí-bi-ma' in toks or 'qi-bi-ma' in toks or 'qí-bi₄-ma' in toks:
        try:
            qibi_i = next(i for i, t in enumerate(toks)
                          if t in ('qí-bi-ma', 'qi-bi-ma', 'qí-bi₄-ma'))
            umma_i = next(i for i, t in enumerate(toks)
                          if t in ('um-ma', 'umma'))
            if qibi_i > umma_i:
                sender_tok = toks_orig[umma_i + 1] if umma_i + 1 < len(toks_orig) else '<gap>'
                recipient_tok = toks_orig[qibi_i - 1] if qibi_i > 0 else '<gap>'
            else:
                recipient_tok = toks_orig[qibi_i - 1] if qibi_i > 0 else '<gap>'
                sender_tok = toks_orig[umma_i + 1] if umma_i + 1 < len(toks_orig) else '<gap>'
            sender = sender_tok.rstrip('-ma').rstrip('ma') if sender_tok.endswith('ma') else sender_tok
            recipient_name = ONO_SPELLING_MAP.get(recipient_tok.lower(), recipient_tok)
            sender_name = ONO_SPELLING_MAP.get(sender.lower(), sender)
            return ('letter_opening', {
                'prefix': f"To {recipient_name}, say: Thus says {sender_name}.",
                'locked_indices': set(range(min(qibi_i, umma_i), max(qibi_i, umma_i) + 2))
            })
        except StopIteration:
            pass

    # ── 2. DEBT FORMULA ───────────────────────────────────────────
    if 'i-ṣé-er' in toks or 'i-se-er' in toks:
        try:
            iser_i = next(i for i, t in enumerate(toks)
                          if t in ('i-ṣé-er', 'i-se-er', 'i-ṣe-er'))
            ishu_i = next(i for i, t in enumerate(toks)
                          if t in ('i-šu', 'i-su', 'ishu') and i > iser_i)
            amount_part = ' '.join(toks_orig[:iser_i])
            return ('debt', {
                'prefix': f"{amount_part} against <gap> owes.",
                'locked_indices': set(range(0, ishu_i + 1))
            })
        except StopIteration:
            pass

    # ── 3. PAYMENT FORMULA ────────────────────────────────────────
    if toks and toks[-1] in ('i-ša-qal', 'i-sa-qal', 'isaqal'):
        try:
            ana_i = next(i for i, t in enumerate(toks) if t in ('a-na', 'ana'))
            duration_part = ' '.join(toks_orig[ana_i + 1:-1])
            return ('payment', {
                'prefix': f"he will pay within {duration_part}.",
                'locked_indices': set(range(ana_i, len(toks)))
            })
        except StopIteration:
            pass

    # ── 4. PENALTY FORMULA ────────────────────────────────────────
    has_shuma = any(t in ('šu-ma', 'su-ma', 'shuma') for t in toks)
    has_usaab = any(t in ('ú-ṣa-áb', 'u-sa-ab', 'usaab') for t in toks)
    if has_shuma and has_usaab:
        usaab_i = next(i for i, t in enumerate(toks)
                       if t in ('ú-ṣa-áb', 'u-sa-ab', 'usaab'))
        return ('penalty', {
            'prefix': f"if he does not pay, he will add interest.",
            'locked_indices': set(range(0, usaab_i + 1))
        })

    return (None, None)


def _fuzzy_onomasticon_lookup(tok, onomasticon_map, threshold=0.85):
    from difflib import SequenceMatcher
    tok_clean = tok.lower().replace('…', '').replace('„', '').replace('{', '').replace('}', '')
    if len(tok_clean) < 4:
        return None
    best_score = 0
    best_name = None
    for spelling, canonical in onomasticon_map.items():
        if abs(len(tok_clean) - len(spelling)) > 4:
            continue
        score = SequenceMatcher(None, tok_clean, spelling).ratio()
        if score > best_score:
            best_score = score
            best_name = canonical
    return best_name if best_score >= threshold else None


# ═══════════════════════════════════════════════════════════════════════
# POST-PROCESSING: clean up English output sequence
# ═══════════════════════════════════════════════════════════════════════

_ILLEGAL_BIGRAMS = {
    ('to', 'to'), ('the', 'the'), ('and', 'and'), ('in', 'in'),
    ('of', 'of'), ('a', 'a'), ('not', 'not'), ('if', 'if'),
    ('our', 'our'), ('this', 'this'), ('that', 'that'),
}

_MEDIAL_PARTICLES = {'verily', 'not', 'indeed', 'truly'}


def _dedup_consecutive(words):
    """Collapse consecutive identical words."""
    if not words:
        return words
    result = [words[0]]
    for w in words[1:]:
        if w.lower() != result[-1].lower():
            result.append(w)
    return result


def _collapse_illegal_bigrams(words):
    """Remove the second word in illegal bigrams."""
    if len(words) < 2:
        return words
    result = [words[0]]
    for i in range(1, len(words)):
        pair = (words[i-1].lower(), words[i].lower())
        if pair not in _ILLEGAL_BIGRAMS:
            result.append(words[i])
    return result


def _dedup_repeated_phrases(words, ngram_size=3):
    """Remove repeated trigram phrases, keeping only the first occurrence."""
    if len(words) < ngram_size * 2:
        return words
    result = list(words)
    seen_ngrams = set()
    i = 0
    while i <= len(result) - ngram_size:
        ngram = tuple(w.lower() for w in result[i:i+ngram_size])
        if ngram in seen_ngrams:
            del result[i:i+ngram_size]
        else:
            seen_ngrams.add(ngram)
            i += 1
    return result


def _fix_particle_position(words):
    """Move sentence-final discourse particles one position left."""
    if len(words) < 2:
        return words
    result = list(words)
    if result[-1].lower() in _MEDIAL_PARTICLES and result[-2].lower() not in _MEDIAL_PARTICLES:
        result[-1], result[-2] = result[-2], result[-1]
    return result


def _post_process_sequence(seq):
    """Apply all post-processing cleanups to a word sequence."""
    words = list(seq)
    words = _dedup_consecutive(words)
    words = _collapse_illegal_bigrams(words)
    words = _dedup_repeated_phrases(words)
    words = _fix_particle_position(words)
    # Strip leading/trailing parentheses from individual words
    words = [re.sub(r'^\(+|\)+$', '', w) for w in words]
    # Remove empty strings left behind
    words = [w for w in words if w.strip()]
    return words


# ═══════════════════════════════════════════════════════════════════════
# COMPOSE SEQUENCE (Beam Search)
# ═══════════════════════════════════════════════════════════════════════

def compose_sequence(akk_tokens, constructions, root_mapping, cluster_mapping,
                     G, bigram_counts, eng_word_to_cluster, akk_token_to_cluster,
                     token_to_zone=None, zone_profiles=None,
                     honeycomb_sub_zones=None):
    """
    Beam search composition with forward-backward validation.
    Returns (output, routes).

    Builds eng_pos_to_akk_tok alignment map during segment assembly
    (after SVO reorder) and passes it into _rrp_fill_gaps.
    """
    akk_feats = [decompose_token(tok) for tok in akk_tokens]
    processed_indices = set()
    routes = []

    # Step 1: Sort constructions by span width (largest first)
    constructions_sorted = sorted(
        [c for c in constructions if 'start_idx' in c and 'end_idx' in c],
        key=lambda c: c.get('end_idx', 0) - c.get('start_idx', 0),
        reverse=True
    )
    for const in constructions_sorted:
        start = const.get('start_idx', 0)
        end   = const.get('end_idx', 0)
        if any(idx in processed_indices for idx in range(start, end)):
            continue
        type_key = f"{const['type']}_{const['pattern']}"
        if type_key in construction_templates:
            template = construction_templates[type_key]
            filled   = template
            slots    = const.get('slots', {})
            for slot, value in slots.items():
                if slot == 'sem_root':
                    continue
                top_candidates = (
                    get_top_eng_words(value, root_mapping, 5) or
                    get_top_eng_words(slots.get('sem_root'), root_mapping, 5) or
                    get_top_eng_words(value, cluster_mapping, 5)
                )
                word = top_candidates[0][0] if top_candidates else value
                if word == value and ('-' in value or not value.isascii()):
                    word = '<gap>'
                filled = filled.replace(f"{{{slot}}}", word)
            processed_indices.update(range(start, end))

    # Step 2: Build segment_entries
    # Each entry: (cands, feats, route_tuple, akk_span_tokens)
    segment_entries = []
    i = 0
    while i < len(akk_tokens):
        if i in processed_indices:
            const = next((c for c in constructions_sorted
                          if c.get('start_idx') == i), None)
            if const:
                type_key = f"{const['type']}_{const['pattern']}"
                if type_key in construction_templates:
                    template    = construction_templates[type_key]
                    filled      = template
                    slot_routes = []
                    for slot, value in const.get('slots', {}).items():
                        if slot == 'sem_root':
                            continue
                        top_candidates = (
                            get_top_eng_words(value, root_mapping, 5) or
                            get_top_eng_words(const.get('slots', {}).get('sem_root'),
                                              root_mapping, 5) or
                            get_top_eng_words(value, cluster_mapping, 5)
                        )
                        word = top_candidates[0][0] if top_candidates else value
                        if word == value and ('-' in value or not value.isascii()):
                            word = '<gap>'
                        src = "ROOT" if value in root_mapping else "CLUSTER"
                        slot_routes.append(f"{value} -> {src} -> {word}")
                        filled = filled.replace(f"{{{slot}}}", word)
                    span_tokens = akk_tokens[i:const.get('end_idx', i + 1)]
                    route_str   = f"CONSTRUCTION[{type_key}]: {' | '.join(slot_routes)}"
                    route_tuple = (" ".join(span_tokens), route_str, filled)
                    segment_entries.append((
                        [(filled, 1.0)],
                        akk_feats[i:const.get('end_idx', i + 1)],
                        route_tuple,
                        span_tokens,
                    ))
                i = const.get('end_idx', i + 1)
                continue
            i += 1
            continue

        feat    = akk_feats[i]
        tok     = akk_tokens[i]
        root    = feat['cad_root'] or feat['root'] or tok
        cluster = akk_token_to_cluster.get(tok, -1)
        root_cands = (get_top_eng_words(feat.get('cad_root'), root_mapping, 5) or
                      get_top_eng_words(feat['root'], root_mapping, 5))

        if feat.get('english'):
            cands     = [(feat['english'], 1.0)]
            route_str = f"DIRECT(LOGO/EBL): {tok} -> {feat['english']}"
        else:
            cluster_cands = get_top_eng_words(cluster, cluster_mapping, 5) if cluster != -1 else []
            cands = root_cands or cluster_cands
            if cands:
                eng_word  = cands[0][0]
                src       = "ROOT" if root_cands else "CLUSTER"
                route_str = f"{src}: {tok} -> {root if root_cands else f'c{cluster}'} -> {eng_word}"
            else:
                # Graph lookup fallback
                graph_key = feat.get('cad_root') or feat.get('sem_root') or _token_to_root(tok)
                try:
                    if graph_key not in G:
                        raise KeyError(f"Root '{graph_key}' not in graph")
                    path_lengths = nx.single_source_dijkstra_path_length(
                        G, graph_key,
                        weight=lambda u, v, d: -np.log(d.get('weight', 1e-6) + 1e-6)
                    )
                    eng_words = {n: path_lengths[n]
                                 for n in path_lengths
                                 if G.nodes[n].get('type') == 'eng_word'}
                    if eng_words:
                        best      = nlargest(5, eng_words.items(), key=lambda x: -x[1])
                        cands     = [(n, np.exp(-abs(v))) for n, v in best]
                        eng_word  = cands[0][0]
                        path_str  = _get_graph_path(G, graph_key, eng_word)
                        route_str = (f"GRAPH: {tok} -> {graph_key} -> {path_str}"
                                     if path_str else
                                     f"GRAPH: {tok} -> {graph_key} -> {eng_word}")
                    else:
                        fuzzy_eng, fuzzy_match = _fuzzy_lookup(tok)
                        if fuzzy_eng:
                            cands     = [(fuzzy_eng, 0.4)]
                            route_str = f"FUZZY: {tok} ~= {fuzzy_match} -> {fuzzy_eng}"
                        else:
                            cands     = [('<gap>', 0.3)]
                            route_str = f"PASSTHROUGH -> <gap>: {tok} (unknown)"
                except Exception:
                    fuzzy_eng, fuzzy_match = _fuzzy_lookup(tok)
                    if fuzzy_eng:
                        cands     = [(fuzzy_eng, 0.4)]
                        route_str = f"FUZZY: {tok} ~= {fuzzy_match} -> {fuzzy_eng}"
                    else:
                        # Check onomasticon before giving up
                        ono_match = _fuzzy_onomasticon_lookup(tok, ONO_SPELLING_MAP)
                        if ono_match:
                            cands     = [(ono_match, 0.6)]
                            route_str = f"ONOMASTICON: {tok} -> {ono_match}"
                        else:
                            cands     = [('<gap>', 0.1)]
                            route_str = f"PASSTHROUGH: {tok} (preserved)"

            if feat['suffix'] == '-ma' and cands and not str(cands[0][0]).startswith('and '):
                cands = [("and " + str(w), p) for w, p in cands]

        route_tuple = (tok, route_str, cands[0][0] if cands else tok)
        segment_entries.append((cands, [feat], route_tuple, [tok]))
        i += 1

    # Step 2.4: Base+suffix deduplication
    # When token[i] is base form and token[i+1] is base+suffix (e.g. -ma),
    # and both have EBL translations, the suffix form already contains
    # the base meaning. Suppress the base to avoid "trading post and to
    # the trading post" repetition.
    deduped = []
    for idx in range(len(segment_entries)):
        if idx < len(segment_entries) - 1:
            curr_tok = segment_entries[idx][3][0]   # Akkadian token
            next_tok = segment_entries[idx+1][3][0]
            curr_eng = str(segment_entries[idx][0][0][0]) if segment_entries[idx][0] else ''
            next_eng = str(segment_entries[idx+1][0][0][0]) if segment_entries[idx+1][0] else ''
            # Check: next token starts with current token (base+suffix)
            # AND next translation contains current translation
            if (next_tok.startswith(curr_tok) and len(next_tok) > len(curr_tok)
                    and curr_eng and curr_eng != '<gap>'
                    and curr_eng.lower() in next_eng.lower()):
                _log(f"[DEDUP] Suppressing '{curr_tok}' -> '{curr_eng}' "
                     f"(contained in '{next_tok}' -> '{next_eng}')", level=1)
                continue
        deduped.append(segment_entries[idx])
    segment_entries = deduped

    # Step 2.5: Molecular folding (replaces mechanical SOV→SVO flip)
    # Energy-minimized reordering: tries permutations, picks lowest-energy fold
    segment_entries = molecular_reorder(segment_entries, bigram_counts)

    # Build segments and routes from reordered entries
    segments = [(cands, feats, span_toks) for cands, feats, _, span_toks in segment_entries]
    routes   = [rt for _, _, rt, _ in segment_entries]

    # Step 3: Beam search (with molecular scoring)
    beams   = [(0.0, [], [], [])]   # (score, eng_seq, akk_seq, monomers)
    max_len = int(len(akk_tokens) * MAX_LENGTH_FACTOR)

    for seg_idx, (cands, seg_akk_feats, span_toks) in enumerate(segments):
        akk_tok = span_toks[0] if span_toks else None
        seg_mono = monomer_profile(seg_akk_feats[0]) if seg_akk_feats else monomer_profile({})
        new_beams = []
        for prev_score, prev_seq, prev_akk, prev_monos in beams:
            for cand_word, cand_prob in cands:
                new_words = (cand_word.split()
                             if isinstance(cand_word, str) and ' ' in cand_word
                             else [cand_word])
                new_seq    = prev_seq + new_words
                new_akk    = prev_akk + [akk_tok] * len(new_words)
                new_monos  = prev_monos + [seg_mono]
                new_score  = prev_score + np.log(cand_prob + 1e-6)

                # Composite similarity score
                if len(new_seq) >= 2:
                    new_score += score_partial_sequence(
                        new_seq, seg_akk_feats, bigram_counts, vectorizer)

                # Molecular interaction energy with previous token
                if len(prev_monos) >= 1:
                    prev_eng = prev_seq[-1] if prev_seq else None
                    curr_eng = new_words[0] if new_words else None
                    pair_e = bond_energy_pair(
                        prev_monos[-1], seg_mono, distance=1,
                        bigram_counts=bigram_counts,
                        eng_word_i=prev_eng, eng_word_j=curr_eng
                    )
                    new_score += pair_e

                # Valence stability bonus/penalty
                new_score += valence_beam_penalty(new_monos)

                new_beams.append((new_score, new_seq, new_akk, new_monos))
        beams = nlargest(BEAM_WIDTH, new_beams, key=lambda x: x[0])
        if beams and len(beams[0][1]) > max_len:
            break

    # Step 4: Forward-backward validation
    best_seqs = [seq for _, seq, _, _ in beams]
    if not best_seqs:
        best_seq = ['<gap>']
    else:
        valid_seqs = [seq for seq in best_seqs
                      if forward_backward_validate(' '.join(seq), akk_tokens, root_mapping)]
        best_seq   = valid_seqs[0] if valid_seqs else best_seqs[0]

    # Build alignment map from segment_entries (post-reorder)
    eng_pos_to_akk_tok  = {}
    eng_pos_to_akk_feat = {}
    cursor = 0
    for cands, seg_feats, _, span_tokens in segment_entries:
        if cursor >= len(best_seq):
            break
        expected = str(cands[0][0]) if cands and cands[0][0] else ''
        expected_words = expected.split() if expected else ['']
        wc = len(expected_words)
        if cursor + wc <= len(best_seq):
            actual_slice = best_seq[cursor:cursor + wc]
            if actual_slice != expected_words and len(cands) > 1:
                for alt_word, _ in cands[1:]:
                    alt_words = str(alt_word).split() if alt_word else ['']
                    alt_wc = len(alt_words)
                    if cursor + alt_wc <= len(best_seq) and best_seq[cursor:cursor + alt_wc] == alt_words:
                        wc = alt_wc
                        break
        wc = max(1, min(wc, len(best_seq) - cursor))
        for offset in range(wc):
            eng_pos_to_akk_tok[cursor + offset] = span_tokens[0] if span_tokens else None
            eng_pos_to_akk_feat[cursor + offset] = seg_feats[0] if seg_feats else {}
        cursor += wc

    # ── Repeat n-gram cleanup ────────────────────────────────
    def _collapse_repeats(seq, min_n=2, max_n=6):
        result = list(seq)
        for n in range(max_n, min_n - 1, -1):
            i = 0
            while i <= len(result) - 2 * n:
                ngram = result[i:i+n]
                if result[i+n:i+2*n] == ngram:
                    _log(f"[DEDUP] Collapsed repeated {n}-gram: {ngram}", level=1)
                    result = result[:i+n] + result[i+2*n:]
                else:
                    i += 1
        return result

    best_seq = _collapse_repeats(best_seq)

    _log(f"[PRE-RRP] best_seq: {best_seq}", level=1)

    global _pre_rrp_snapshots
    globals()['_pre_rrp_snapshots'].append({
        'best_seq':           list(best_seq),
        'akk_tokens':         list(akk_tokens),
        'akk_feats':          list(akk_feats),
        'eng_pos_to_akk_tok': dict(eng_pos_to_akk_tok),
        'eng_pos_to_akk_feat': dict(eng_pos_to_akk_feat),
    })

    # Step 5: RRP gap-fill with correct alignment map
    best_seq = _rrp_fill_gaps(
        best_seq, akk_tokens, akk_feats, G,
        eng_word_to_cluster, akk_token_to_cluster,
        eng_pos_to_akk_tok,
        eng_pos_to_akk_feat,
        bigram_counts=bigram_counts,
        honeycomb_sub_zones=honeycomb_sub_zones,
        root_to_eng_words=root_mapping
    )

    # Step 6: Post-processing
    best_seq = _post_process_sequence(best_seq)

    # Step 7: Surviving <gap> tokens stay as <gap> — transliteration fallback removed
    # (raw Akkadian in output scores zero BLEU; <gap> is neutral)
    pass

    output = ' '.join(best_seq).capitalize() + '.'
    return output, routes


# ═══════════════════════════════════════════════════════════════════════
# RRP GAP-FILLER
# ═══════════════════════════════════════════════════════════════════════

def _rrp_fill_gaps(best_seq, akk_tokens, akk_feats, G,
                   eng_word_to_cluster, akk_token_to_cluster,
                   eng_pos_to_akk_tok=None,
                   eng_pos_to_akk_feat=None,
                    bigram_counts=None,
                   honeycomb_sub_zones=None,
                   root_to_eng_words=None):
                   
    """
    RRP gap-filler with corrected alignment.

    Uses the alignment map built AFTER reordering in compose_sequence.
    Falls back to legacy reconstruction only if map not provided (backward compat).
    """
    result = list(best_seq)
    WINDOW         = 2
    MIN_CONFIDENCE = 0.3

    # ── Build alignment map ───────────────────────────────────────
    if eng_pos_to_akk_tok is not None:
        eng_to_akk      = eng_pos_to_akk_tok
        eng_to_akk_feat = eng_pos_to_akk_feat or {}
    else:
        # Legacy fallback (backward compat only)
        eng_to_akk      = {}
        eng_to_akk_feat = {}
        eng_idx = 0
        for akk_idx, tok in enumerate(akk_tokens):
            root = _token_to_root(tok)
            feat = decompose_token(tok)
            base = feat.get('root') or feat.get('sem_root')
            root_ac = akk_token_to_cluster.get(root, -1) if root else -1
            base_ac = akk_token_to_cluster.get(base, -1) if base else -1
            ac      = root_ac if root_ac != -1 else base_ac
            _log(f"[RRP-FALLBACK] tok={tok} root={root}->{root_ac} base={base}->{base_ac}", level=2)
            if eng_idx < len(result):
                eng_to_akk[eng_idx]      = tok
                eng_to_akk_feat[eng_idx] = akk_feats[akk_idx]
                feat_english = akk_feats[akk_idx].get('english', '') or ''
                word_count   = len(feat_english.split()) if feat_english else 1
                eng_idx     += max(1, word_count)

    # ── Gap-fill loop ─────────────────────────────────────────────
    for i, word in enumerate(result):
        if word != '<gap>':
            continue

        _log(f"[RRP] Gap at position {i}", level=2)
        tok = eng_to_akk.get(i)
        _log(f"[RRP] Mapped token: {tok}", level=2)
        if not tok:
            _log(f"[RRP] No token mapped to position {i} — skipping", level=1)
            continue
        _log(f"[RRP] tok={tok} ac={akk_token_to_cluster.get(tok, -1)}", level=1)

        # Resolve cluster for this token
        ac = akk_token_to_cluster.get(tok, -1)
        root = _token_to_root(tok)
        feat = eng_to_akk_feat.get(i) or decompose_token(tok)
        base = feat.get('root') or feat.get('sem_root')
        if ac == -1:
            root_ac = akk_token_to_cluster.get(root, -1) if root else -1
            base_ac = akk_token_to_cluster.get(base, -1) if base else -1
            ac = root_ac if root_ac != -1 else base_ac
        if ac == -1:
            t_norm_key = (tok.lower()
                          .replace('í','i').replace('ú','u')
                          .replace('á','a').replace('é','e')
                          .replace('\u1e2b','h').replace('\u1e25','h'))
            ac = akk_token_to_cluster.get(t_norm_key, -1)

        akk_node = f'uc{ac}' if ac != -1 else None

        # ── Strategy 1: Walk graph from root/base/tok nodes ───────
        candidates = {}
        for node_key in [root, base, tok]:
            if node_key and node_key in G:
                for cell in G.successors(node_key):
                    for eng in G.successors(cell):
                        # Skip cluster node IDs — only collect actual English words
                        if isinstance(eng, str) and not eng.startswith('uc'):
                            candidates[eng] = candidates.get(eng, 0) + 1

        if ac == -1 and not candidates:
            pass  # no graph hits and no cluster — fall through to Mendeleev

        if candidates and bigram_counts:
            prev_word = result[i-1] if i > 0 else None
            next_word = result[i+1] if i < len(result)-1 else None
            scored = {}
            for eng, count in candidates.items():
                score = count
                if prev_word and (prev_word, eng) in bigram_counts:
                    score += bigram_counts[(prev_word, eng)] * 2
                if next_word and (eng, next_word) in bigram_counts:
                    score += bigram_counts[(eng, next_word)] * 2
                scored[eng] = score
            result[i] = max(scored, key=scored.get)
            _log(f"[RRP-ROOT] {tok} -> {result[i]}", level=1)
            continue

        # ── Strategy 2: Fuzzy fallback (when no graph node) ───────
        if not akk_node or akk_node not in G:
            fuzzy_eng, _ = _fuzzy_lookup(tok)
            if fuzzy_eng:
                _log(f"[RRP-FUZZY] {tok} -> {fuzzy_eng}", level=1)
                result[i] = fuzzy_eng
                continue

            # ── Strategy 2b: MENDELEEV PREDICTION ─────────────────
            # Token has no cluster, no graph node, no fuzzy match.
            # Predict from the gap's chemical context:
            #   1. Neighbor interpolation on SOM grid
            #   2. Valence constraint (what POS must fill this slot?)
            #   3. Charge balance (does the sentence need + or - here?)
            #
            # Like Mendeleev predicting germanium: we know the gap's
            # row and column in the periodic table from its neighbors.

            # 1. Collect neighbor clusters — the gap's "period and group"
            neighbor_clusters = []
            neighbor_monomers = []
            for offset in range(-2, 3):
                if offset == 0:
                    continue
                j = i + offset
                if 0 <= j < len(result) and result[j] != '<gap>':
                    nb_word = result[j]
                    nb_ec = eng_word_to_cluster.get(nb_word, -1)
                    if nb_ec != -1:
                        neighbor_clusters.append(nb_ec)
                    # Get monomer of neighbor's aligned Akkadian token
                    nb_akk_feat = eng_to_akk_feat.get(j)
                    if nb_akk_feat:
                        neighbor_monomers.append(monomer_profile(nb_akk_feat))

            if not neighbor_clusters:
                _log(f"[RRP] Node {akk_node} not in graph, no neighbor clusters — skipping", level=1)
                continue

            # 2. Valence constraint: what POS should fill this gap?
            #    Left neighbor = verb → gap is likely ACC noun (object)
            #    Left neighbor = NOM noun → gap is likely verb or GEN noun
            #    Both neighbors = nouns → gap is likely verb
            expected_pos = None
            if neighbor_monomers:
                left_mono  = neighbor_monomers[0] if len(neighbor_monomers) > 0 else None
                right_mono = neighbor_monomers[-1] if len(neighbor_monomers) > 1 else None

                if left_mono and left_mono.get('is_verb'):
                    expected_pos = 'NOUN'
                elif left_mono and left_mono['pos'] == 'NOUN' and right_mono and right_mono['pos'] == 'NOUN':
                    expected_pos = 'VERB'
                elif left_mono and left_mono['pos'] == 'NOUN':
                    expected_pos = 'VERB'
                # NOTE: No charge balance — Akkadian sentences are not
                # charge-neutral. Seal inscriptions are all NOM/GEN,
                # contracts lean ACC. Ions exist. Don't force balance.

            # 3. Interpolate: scan English words in neighbor clusters
            #    These are the "predicted properties" of the missing element
            mendeleev_candidates = {}
            for nc in neighbor_clusters:
                # Check if this cluster's zone has honeycomb refinement
                honeycomb_hit = False
                if honeycomb_sub_zones:
                    for z_id, sub_zones in honeycomb_sub_zones.items():
                        for sub_key, sub_toks in sub_zones.items():
                            # Check if any token in this sub-zone maps to this cluster
                            for st in sub_toks:
                                if akk_token_to_cluster.get(st, -1) == nc:
                                    # Use sub-zone tokens for candidates instead
                                    for st2 in sub_toks:
                                        for eng_pair in root_to_eng_words.get(st2, []):
                                            eng = eng_pair[0] if isinstance(eng_pair, tuple) else eng_pair
                                            if eng.lower() in ('the', 'a', 'and', 'to', 'in', 'of', 'for', 'is'):
                                                continue
                                            mendeleev_candidates[eng] = mendeleev_candidates.get(eng, 0) + 1
                                    honeycomb_hit = True
                                    break
                            if honeycomb_hit:
                                break
                        if honeycomb_hit:
                            break

                if not honeycomb_hit:
                    # Fallback: original cluster lookup
                    nc_node = f'uc{nc}'
                    if nc_node in G:
                        for eng in G.successors(nc_node):
                            if G.nodes.get(eng, {}).get('type') == 'eng_word':
                                if eng.lower() in ('the', 'a', 'and', 'to', 'in', 'of', 'for', 'is'):
                                    continue
                                mendeleev_candidates[eng] = mendeleev_candidates.get(eng, 0) + 1
            if mendeleev_candidates:
                # Score by: frequency in neighbor zones + bigram fit + charge match
                scored = {}
                for cand, freq in mendeleev_candidates.items():
                    score = freq
                    # Bigram coherence with actual neighbors
                    if bigram_counts:
                        prev_w = result[i-1].lower() if i > 0 else None
                        next_w = result[i+1].lower() if i < len(result)-1 else None
                        if prev_w and (prev_w, cand.lower()) in bigram_counts:
                            score += bigram_counts[(prev_w, cand.lower())] * 3
                        if next_w and (cand.lower(), next_w) in bigram_counts:
                            score += bigram_counts[(cand.lower(), next_w)] * 3
                    scored[cand] = score

                if scored:
                    best_mendeleev = max(scored, key=scored.get)
                    best_m_score = scored[best_mendeleev]
                    # Silver stain: candidate must beat silver baseline
                    silver_score = 0.0
                    if bigram_counts:
                        prev_w = result[i-1].lower() if i > 0 else None
                        next_w = result[i+1].lower() if i < len(result)-1 else None
                        silver_score = 37.5  # corpus baseline
                        if prev_w and (prev_w, 'silver') in bigram_counts:
                            silver_score += bigram_counts[(prev_w, 'silver')] * 3
                        if next_w and ('silver', next_w) in bigram_counts:
                            silver_score += bigram_counts[('silver', next_w)] * 3

                    # ── COOC silver discount ─────────────────────────────
                    # If training data directly attests this token -> best_mendeleev,
                    # discount the silver baseline. Token-level evidence > corpus noise.
                    if 'COOC_TABLE' in globals() and best_mendeleev.lower() != 'silver':
                        _tn2 = tok.lower().replace('í','i').replace('ú','u').replace('á','a').replace('é','e')
                        _tn2 = re.sub(r'[\u201e+\u2026"\'()]', '', _tn2)
                        _cooc2 = COOC_TABLE.get(_tn2, {})
                        _ctotal = COOC_TOTAL.get(_tn2, 0)
                        if _cooc2 and _ctotal > 0:
                            _hits = _cooc2.get(best_mendeleev, 0)
                            if _hits == 0 and len(best_mendeleev) >= 4:
                                _hits = sum(v for k,v in _cooc2.items() if k.startswith(best_mendeleev[:4]))
                            if _hits > 0:
                                _conf = min(1.0, (_hits / _ctotal) * 5)
                                _old_silver = silver_score
                                silver_score *= (1.0 - _conf * 0.8)
                                _log(f"[RRP-COOC-DISCOUNT] {tok} -> {best_mendeleev} "
                                     f"cooc_conf={_conf:.2f} silver: {_old_silver:.1f} -> {silver_score:.1f}", level=1)
                    if best_m_score >= 1.0 and best_m_score > silver_score:
                        result[i] = best_mendeleev
                        pos_tag = expected_pos or '?'
                        _log(f"[RRP-MENDELEEV] {tok} -> {best_mendeleev} "
                             f"(score={best_m_score:.1f} > silver={silver_score:.1f}, "
                             f"expected_pos={pos_tag}, "
                             f"neighbor_clusters={neighbor_clusters})", level=1)
                        continue
                    elif best_m_score >= 1.0:
                        _log(f"[RRP-STAIN] {tok} -> WASHED "
                             f"({best_mendeleev}={best_m_score:.1f} < silver={silver_score:.1f})",
                             level=1)
                        continue

            _log(f"[RRP] Node {akk_node} not in graph, Mendeleev found no candidates — skipping", level=1)
            continue

        # ── Strategy 3: Dijkstra neighbor-context scoring ─────────
        neighbors = []
        for offset in range(-WINDOW, WINDOW + 1):
            if offset == 0:
                continue
            j = i + offset
            if 0 <= j < len(result) and result[j] != '<gap>':
                neighbors.append(result[j])

        if not neighbors:
            continue

        neighbor_nodes = set()
        for n in neighbors:
            ec = eng_word_to_cluster.get(n, -1)
            if ec != -1:
                neighbor_nodes.add(f'uc{ec}')
            if n in G:
                neighbor_nodes.add(n)

        if not neighbor_nodes:
            continue

        try:
            path_lengths = nx.single_source_dijkstra_path_length(
                G, akk_node,
                weight=lambda u, v, d: -np.log(d.get('weight', 1e-6) + 1e-6),
                cutoff=10
            )
        except Exception:
            continue

        eng_candidates = {
            n: path_lengths[n]
            for n in path_lengths
            if G.nodes.get(n, {}).get('type') == 'eng_word'
        }

        if not eng_candidates:
            fuzzy_eng, _ = _fuzzy_lookup(tok)
            if fuzzy_eng:
                result[i] = fuzzy_eng
            continue

        # Re-score by neighbor coherence
        scored = []
        for cand_word, dist_score in eng_candidates.items():
            coherence  = 0.0
            cand_ec    = eng_word_to_cluster.get(cand_word, -1)
            cand_node  = f'uc{cand_ec}' if cand_ec != -1 else None
            for nb_node in neighbor_nodes:
                if cand_node and G.has_edge(nb_node, cand_node):
                    coherence += G[nb_node][cand_node].get('weight', 0)
                elif G.has_edge(nb_node, cand_word):
                    coherence += G[nb_node][cand_word].get('weight', 0)
            final_score = np.exp(-abs(dist_score)) + coherence
            scored.append((cand_word, final_score))

        if not scored:
            continue

        best_cand, best_score = max(scored, key=lambda x: x[1])
        if best_score >= MIN_CONFIDENCE:
            _log(f"[RRP] Filled gap at {i}: '{tok}' -> '{best_cand}' (score={best_score:.3f})", level=1)
            result[i] = best_cand

    return result


# ═══════════════════════════════════════════════════════════════════════
# MAIN
# ═══════════════════════════════════════════════════════════════════════

def main():
    global _root_mapping, _cluster_mapping, _test_df
    global akk_token_to_cluster, vectorizer
    t0 = time.time()

    # Load data
    train_df = pd.read_csv(DATA_DIR / "train.csv")
    test_df  = pd.read_csv(DATA_DIR / "test.csv")

    translit_col = "transliteration" if "transliteration" in train_df.columns else train_df.columns[1]
    trans_col    = "translation" if "translation" in train_df.columns else (
        train_df.columns[2] if len(train_df.columns) > 2 else "translation"
    )


    # ── TRAIN DATA CLEANING ─────────────────────────────────────────
    REMOVE_TOKENS = ['fem.', 'sing.', 'pl.', 'plural', '(?)', '<<', '>>', 'xx']
    FRAC_UNICODE = {
        '1/2': '½', '1/3': '⅓', '1/4': '¼', '2/3': '⅔',
        '1/6': '⅙', '5/6': '⅚', '3/4': '¾', '5/8': '⅝',
        '1/8': '⅛', '3/8': '⅜', '7/8': '⅞',
    }
    def decimal_to_fraction(d_str):
        try:
            from fractions import Fraction
            f = Fraction(d_str).limit_denominator(12)
            whole = int(f)
            frac = f - whole
            if frac == 0:
                return str(whole)
            char = FRAC_UNICODE.get(f"{frac.numerator}/{frac.denominator}")
            if char is None:
                return d_str
            return f"{whole}{char}" if whole else char
        except:
            return d_str
    SUB_MAP = str.maketrans('₀₁₂₃₄₅₆₇₈₉', '0123456789')
    
    for idx, row in train_df.iterrows():
        trans = str(row[trans_col])
        translit = str(row[translit_col])
    
        # Translation: strip annotation artifacts
        for tok in REMOVE_TOKENS:
            trans = trans.replace(tok, '')
        # Strip sentence-final periods and colons from words
        trans = re.sub(r'(?<=[a-zA-Z])\.(?=\s|$)', '', trans)
        trans = re.sub(r':(?=\s|$)', '', trans)
        # Slash alternatives — keep first option
        trans = re.sub(r'(\S+)\s*/\s*\S+', r'\1', trans)
        # PN literal → <gap>
        trans = re.sub(r'\bPN\b', '<gap>', trans)
        trans = re.sub(r'\bx\b', '<gap>', trans)
        # Curly quotes → straight
        trans = trans.replace('\u201c', '"').replace('\u201d', '"')
        trans = trans.replace('\u2018', "'").replace('\u2019', "'")
    
        # Both: fraction normalization (programmatic — handles all decimals)
        trans    = re.sub(r'\b\d+\.\d+\b', lambda m: decimal_to_fraction(m.group()), trans)
        translit = re.sub(r'\b\d+\.\d+\b', lambda m: decimal_to_fraction(m.group()), translit)
    
        # Transliteration: subscripts and special chars
        translit = translit.translate(SUB_MAP)
        translit = translit.replace('Ḫ', 'H').replace('ḫ', 'h')
    
        # Collapse extra spaces
        trans = re.sub(r'\s+', ' ', trans).strip()
        translit = re.sub(r'\s+', ' ', translit).strip()
    
        train_df.at[idx, trans_col] = trans
        train_df.at[idx, translit_col] = translit
    
    _log(f"[CLEAN] Training data cleaned: annotations stripped, "
         f"fractions normalized, quotes straightened, subscripts converted, Ḫ→H")
        
    # Also clean test transliterations the same way
    for idx, row in test_df.iterrows():
        translit = str(row[translit_col] if translit_col in test_df.columns else row[test_df.columns[1]])
        translit = re.sub(r'\b\d+\.\d+\b', lambda m: decimal_to_fraction(m.group()), translit)
        translit = translit.translate(SUB_MAP)
        translit = translit.replace('Ḫ', 'H').replace('ḫ', 'h')
        test_df.at[idx, test_df.columns[1] if translit_col not in test_df.columns else translit_col] = translit
    
    _log(f"[CLEAN] Test transliterations normalized to match training")
    pub_df = pd.read_csv('/kaggle/input/competitions/deep-past-initiative-machine-translation/published_texts.csv')
    pub_lookup = dict(zip(pub_df['oare_id'], pub_df['transliteration']))
    
    recovered = 0
    for idx, row in train_df.iterrows():
        oid = row['oare_id']
        if oid in pub_lookup:
            pub_translit = str(pub_lookup[oid])
            cur_translit = str(row[translit_col])
            # Suspicious: published version much longer, current looks truncated
            if len(pub_translit) > len(cur_translit) * 1.5 and len(cur_translit) < 200:
                train_df.at[idx, translit_col] = pub_translit
                recovered += 1
    
    _log(f"[CLEAN] Recovered {recovered} truncated transliterations from published_texts.csv")
    _log(f"\n[DATA] train: {len(train_df)} rows | test: {len(test_df)} rows")
    _log(f"[DATA] Columns: translit={translit_col}, trans={trans_col}")
    train_pairs = [(str(row[translit_col]).split(), str(row[trans_col])) for _, row in train_df.iterrows()]

    _log(f"\n[DATA] train: {len(train_df)} rows | test: {len(test_df)} rows")
    _log(f"[DATA] Columns: translit={translit_col}, trans={trans_col}")

    train_pairs = [(str(row[translit_col]).split(), str(row[trans_col])) for _, row in train_df.iterrows()]

    # ── COOC_TABLE: token-level co-occurrence (used for silver-wash discount) ──
    _ENG_STOP_RRP = {
        'the','a','an','and','or','of','to','in','is','are','was',
        'be','it','its','he','she','they','we','you','that','this',
        'these','those','for','on','at','by','with','from','not','so',
        'as','if','but','him','her',
    }
    COOC_TABLE = {}
    COOC_TOTAL = {}
    for _akk_toks, _eng_sent in train_pairs:
        _eng_ws = [w for w in re.findall(r'\b[a-z]{2,}\b', _eng_sent.lower())
                   if w not in _ENG_STOP_RRP]
        for _t in _akk_toks:
            _tn = _t.lower().replace('í','i').replace('ú','u').replace('á','a').replace('é','e')
            _tn = re.sub(r'[\u201e+\u2026"\'()]', '', _tn)
            for _ew in _eng_ws:
                COOC_TABLE.setdefault(_tn, {})
                COOC_TABLE[_tn][_ew] = COOC_TABLE[_tn].get(_ew, 0) + 1
            COOC_TOTAL[_tn] = COOC_TOTAL.get(_tn, 0) + 1
    _log(f"[COMPASS-COOC] Built COOC_TABLE: {len(COOC_TABLE)} tokens from training pairs")
    LEX_PATHS = [
        '/kaggle/input/datasets/alvarochaveste/akkylexyfromdic/akkadian_lexiconfromdick.csv',
        '/kaggle/input/datasets/alvarochaveste/akkylexyfromdic/akkadian_lexiconfromdick1-74.csv',
    ]
    weighted_pairs = build_weighted_pairs(train_pairs, LEX_PATHS)
    akk_corpus = [' '.join(akk) for akk, _ in train_pairs]
    eng_corpus = [eng for _, eng in train_pairs]

    if DEBUG_LEVEL >= 1:
        _log(f"[DATA] Sample train pair:")
        akk, eng = train_pairs[0]
        _log(f"  Akk ({len(akk)} tokens): {' '.join(akk[:8])}{'...' if len(akk)>8 else ''}")
        _log(f"  Eng: {eng[:80]}{'...' if len(eng)>80 else ''}")

    # Joint TF-IDF
    vectorizer = TfidfVectorizer(max_features=5000)
    vectorizer.fit(akk_corpus + eng_corpus)

    # Token sets
    all_akk_tokens = list(set(tok for akk, _ in train_pairs for tok in akk))
    ENG_STOPWORDS = {
        'to', 'the', 'and', 'of', 'in', 'a', 'an', 'is', 'are', 'was',
        'were', 'be', 'been', 'being', 'have', 'has', 'had', 'do', 'does',
        'did', 'will', 'would', 'could', 'should', 'may', 'might', 'shall',
        'not', 'no', 'nor', 'from', 'with', 'at', 'by', 'for', 'on', 'as',
        'into', 'through', 'during', 'before', 'after', 'above', 'below',
        'between', 'out', 'up', 'about', 'against', 'or', 'but', 'if',
        'while', 'that', 'this', 'these', 'those', 'it', 'its', 'he', 'his',
        'she', 'her', 'they', 'their', 'we', 'our', 'you', 'your', 'him',
    }
    all_eng_words = list(set(
        word for _, eng in train_pairs
        for word in eng.split()
        if word.lower() not in ENG_STOPWORDS and len(word) > 2
    ))

    # Unified SOM clustering
    akk_token_to_cluster, eng_word_to_cluster, _, _, paired_vectors, all_akk, som = build_unified_som_clusters(
        weighted_pairs, n_cols=12, n_rows=16
    )

    # Build mappings
    root_to_eng_words        = defaultdict(Counter)
    akk_cluster_to_eng_words = defaultdict(Counter)

    SEED_WEIGHT = 10
    for akk_form, eng_meaning in EBL_LEXICON.items():
        if eng_meaning:
            root_to_eng_words[akk_form][eng_meaning] += SEED_WEIGHT
    for akk_form, eng_meaning in CAD_DICT.items():
        if pd.notna(eng_meaning) and eng_meaning:
            root_to_eng_words[str(akk_form)][str(eng_meaning)] += SEED_WEIGHT
    _log(f"[SEED] root_to_eng_words pre-seeded: {len(root_to_eng_words)} entries")


    eng_doc_freq = Counter()
    for _, eng_sent, _ in weighted_pairs:
        for word in set(eng_sent.lower().split()):
            eng_doc_freq[word] += 1
    n_docs = len(weighted_pairs)

    for akk_tokens_train, eng_sent, w in weighted_pairs:
        eng_words = set(eng_sent.lower().split())
        token_features = [decompose_token(tok) for tok in akk_tokens_train]
        for idx, tok in enumerate(akk_tokens_train):
            cluster   = akk_token_to_cluster.get(tok, -1)
            root      = token_features[idx]['cad_root'] or token_features[idx]['root'] or tok
            base_root = token_features[idx]['root']
            sem_root  = token_features[idx].get('sem_root', base_root)
            for word in eng_words:
                df_weight = max(0.1, n_docs / (1 + eng_doc_freq.get(word, 1)))
                root_to_eng_words[root][word] += w * min(df_weight, 10.0)
                if sem_root != base_root:
                    root_to_eng_words[sem_root][word] += 1
                if word.lower() in ENG_STOPWORDS or len(word) <= 2:
                    continue
                akk_cluster_to_eng_words[cluster][word] += w * min(df_weight, 10.0)
    # Normalize to probabilities
    for mapping in [akk_cluster_to_eng_words, root_to_eng_words]:
        for key in mapping:
            total = sum(mapping[key].values())
            if total > 0:
                for word in mapping[key]:
                    mapping[key][word] /= total

    token_to_zone, zone_profiles, zone_map = build_zone_system(
        akk_token_to_cluster, akk_cluster_to_eng_words,
        n_cols=12, n_rows=16
    )

    # ── HONEYCOMB REFINEMENT ─────────────────────────────────────────
    HONEYCOMB_THRESHOLD = 500
    HONEYCOMB_MIN_POP = 40
    HONEYCOMB_GRID = (4, 4)

# Build token-to-vector lookup for honeycomb retraining
    akk_tok_to_vec = {tok: paired_vectors[i] for i, tok in enumerate(all_akk)
                      if i < len(paired_vectors)}

    honeycomb_sub_zones = {}
    for z_id, profile in zone_profiles.items():
        if profile['token_count'] < HONEYCOMB_THRESHOLD:
            continue

        # Collect tokens and vectors for this zone
        zone_toks = [tok for tok, zid in token_to_zone.items() if zid == z_id]
        zone_vecs = [akk_tok_to_vec[tok] for tok in zone_toks if tok in akk_tok_to_vec]
        zone_toks = [tok for tok in zone_toks if tok in akk_tok_to_vec]

        if not zone_vecs:
            continue

        sub_cells = HONEYCOMB_GRID[0] * HONEYCOMB_GRID[1]
        if len(zone_toks) / sub_cells < HONEYCOMB_MIN_POP:
            continue

        zone_matrix = np.array(zone_vecs)
        mini = MiniSom(
            HONEYCOMB_GRID[0], HONEYCOMB_GRID[1],
            zone_matrix.shape[1],
            sigma=1.0, learning_rate=0.5,
            neighborhood_function='gaussian',
            topology='hexagonal',
            random_seed=42
        )
        mini.random_weights_init(zone_matrix)
        mini.train(zone_matrix, num_iteration=500, verbose=False)

        sub_assignments = defaultdict(list)
        for tok, vec in zip(zone_toks, zone_vecs):
            c, r = mini.winner(vec)
            sub_key = f"{z_id}.{r * HONEYCOMB_GRID[1] + c}"
            sub_assignments[sub_key].append(tok)

        honeycomb_sub_zones[z_id] = dict(sub_assignments)

        _log(f"[HONEYCOMB] Zone {z_id} ({len(zone_toks)} tokens) -> "
             f"{len(sub_assignments)} sub-zones "
             f"(avg {len(zone_toks) // len(sub_assignments)} tokens/sub)")

        if DEBUG_LEVEL >= 1:
            for sub_key, sub_toks in sorted(sub_assignments.items()):
                sub_eng = []
                for t in sub_toks:
                    for eng_pair in root_to_eng_words.get(t, []):
                        if isinstance(eng_pair, tuple):
                            sub_eng.append(eng_pair[0])
                        else:
                            sub_eng.append(eng_pair)
                top_sub = [w for w, _ in Counter(sub_eng).most_common(3)]
                _log(f"    Sub-zone {sub_key}: {len(sub_toks)} tokens | "
                     f"top_eng={top_sub}")

    _log(f"[HONEYCOMB] {len(honeycomb_sub_zones)} zones refined")

    # Bigram counts
    bigram_counts = defaultdict(lambda: 1e-6)
    for _, eng in train_pairs:
        words = eng.lower().split()
        for i in range(len(words) - 1):
            bigram_counts[(words[i], words[i + 1])] += 1
    _log(f"[BIGRAM] {len(bigram_counts)} bigram types from training corpus")

    # Train molecular bond energies from training data
    train_bond_energies(train_pairs, akk_token_to_cluster)

    # Mapping stats
    roots_with_mapping    = sum(1 for k in root_to_eng_words if root_to_eng_words[k])
    clusters_with_mapping = sum(1 for k in akk_cluster_to_eng_words if akk_cluster_to_eng_words[k])
    _log(f"\n[MAPPING] root_to_eng_words: {roots_with_mapping} roots with >=1 English word")
    _log(f"[MAPPING] akk_cluster_to_eng_words: {clusters_with_mapping} clusters with >=1 English word")
    if DEBUG_LEVEL >= 1:
        sample_root = next((r for r in root_to_eng_words if root_to_eng_words[r]), None)
        if sample_root:
            top = root_to_eng_words[sample_root].most_common(3)
            _log(f"[MAPPING] Sample root '{sample_root}' -> {top}")

    # ── Build graph ───────────────────────────────────────────────
    G = nx.DiGraph()
    all_akk_roots = set(_token_to_root(tok) for tok in all_akk_tokens)
    for root in all_akk_roots:
        G.add_node(root, type='akk_root')
    for word in all_eng_words:
        G.add_node(word, type='eng_word')

    N_CLUSTERS = 12 * 16
    for c in range(N_CLUSTERS):
        G.add_node(f'uc{c}', type='unified_cell')

    # Edges: root -> unified_cell
    for tok in all_akk_tokens:
        root = _token_to_root(tok)
        cluster = akk_token_to_cluster.get(tok, -1)
        if cluster != -1:
            G.add_edge(root, f'uc{cluster}', weight=1.0)

    # Edges: unified_cell -> English word
    for word, cluster in eng_word_to_cluster.items():
        G.add_edge(f'uc{cluster}', word, weight=1.0)

    # Hard edges: EBL and CAD as fixed nodes (weight=1.0)
    hard_edge_count = 0
    for akk_form, eng_meaning in {**EBL_LEXICON, **CAD_DICT}.items():
        if not eng_meaning or pd.isna(eng_meaning):
            continue
        akk_form    = str(akk_form)
        eng_meaning = str(eng_meaning)
        if akk_form not in G:
            G.add_node(akk_form, type='akk_root')
        if eng_meaning not in G:
            G.add_node(eng_meaning, type='eng_word')
        G.add_edge(akk_form, eng_meaning, weight=1.0)
        hard_edge_count += 1
    _log(f"[GRAPH] Hard edges added from EBL+CAD: {hard_edge_count}")

    # Topology edges: adjacent SOM cells
    n_grid_cols, n_grid_rows = 12, 16
    adjacency_edge_count = 0
    for r in range(n_grid_rows):
        for c in range(n_grid_cols):
            flat_id = r * n_grid_cols + c
            for dr, dc in [(-1, 0), (1, 0), (0, -1), (0, 1),
                           (-1, 1 if r % 2 == 0 else -1),
                           (1, 1 if r % 2 == 0 else -1)]:
                nr, nc = r + dr, c + dc
                if 0 <= nr < n_grid_rows and 0 <= nc < n_grid_cols:
                    neighbor_id = nr * n_grid_cols + nc
                    G.add_edge(f'uc{flat_id}', f'uc{neighbor_id}', weight=0.5)
                    adjacency_edge_count += 1

    # Cluster-level prob edges
    for ac in akk_cluster_to_eng_words:
        for word, prob in akk_cluster_to_eng_words[ac].items():
            if prob > 0.1:
                G.add_edge(f'uc{ac}', word, weight=prob)


    coocc_edge_count = 0
    max_count = max(bigram_counts.values())
    for (w1, w2), count in bigram_counts.items():
        weight = count / max_count
        if weight >= COOCC_WEIGHT_THRESHOLD:
            if G.has_node(w1) and G.has_node(w2):
                if not G.has_edge(w1, w2):
                    G.add_edge(w1, w2, weight=weight)
                    coocc_edge_count += 1

    # FLOAT UNKNOWN TEST TOKENS into graph
    for tok in test_df['transliteration'].str.split().explode().unique():
        if not G.has_node(tok):
            G.add_node(tok, type='akk_root')
            G.add_edge(tok, 'uc138', weight=0.1)

    n_nodes = G.number_of_nodes()
    n_edges = G.number_of_edges()
    _log(f"\n[GRAPH] Nodes: {n_nodes} | Edges: {n_edges} | topology edges: {adjacency_edge_count}")
    _log(f"[GRAPH] Unified SOM: cross-lingual alignment baked into topology")
    print(f"[GRAPH] Unified topology: {adjacency_edge_count} adjacency edges between SOM cells")

    # ── Translate test set ────────────────────────────────────────
    test_translit = "transliteration" if "transliteration" in test_df.columns else test_df.columns[-1]
    test_id_col   = "id" if "id" in test_df.columns else test_df.columns[0]

    _log(f"\n{'='*70}")
    _log("TRANSLATING TEST SET")
    _log(f"{'='*70}")

    ids = []
    translations = []
    global _pre_rrp_snapshots
    _pre_rrp_snapshots = []

    for test_idx, (_, row) in enumerate(test_df.iterrows()):
        akk_sent_id = row[test_id_col]
        akk_tokens  = str(row[test_translit]).split()

        # ── FORMULA PRE-PASS ──────────────────────────────────────
        formula_type, formula_result = detect_formula(akk_tokens)
        if formula_type is not None:
            locked    = formula_result['locked_indices']
            remainder = [t for i, t in enumerate(akk_tokens) if i not in locked]
            base      = formula_result['prefix']
            if remainder:
                token_features = [decompose_token(tok) for tok in remainder]
                constructions = []
                for idx, feat in enumerate(token_features):
                    consts = identify_construction_types(feat, remainder, idx)
                    for c in consts:
                        if 'start_idx' not in c:
                            c['start_idx'] = idx
                            c['end_idx']   = idx + 1
                    constructions.extend(consts)
                try:
                    rest, routes = compose_sequence(
                        remainder, constructions, root_to_eng_words,
                        akk_cluster_to_eng_words, G, bigram_counts,
                        eng_word_to_cluster, akk_token_to_cluster, honeycomb_sub_zones=honeycomb_sub_zones)
                    output = base + ' ' + rest
                except Exception as exc:
                    print(f"[ERROR] formula sentence {test_idx}: {exc}")
                    output = base
                    routes = []
            else:
                output = base
                routes = []
                constructions = []
        else:
            # ── NORMAL PATH ───────────────────────────────────────
            token_features = [decompose_token(tok) for tok in akk_tokens]
            constructions = []
            for idx, feat in enumerate(token_features):
                consts = identify_construction_types(feat, akk_tokens, idx)
                for c in consts:
                    if 'start_idx' not in c:
                        c['start_idx'] = idx
                        c['end_idx']   = idx + 1
                constructions.extend(consts)
            try:
                output, routes = compose_sequence(
                    akk_tokens, constructions, root_to_eng_words,
                    akk_cluster_to_eng_words, G, bigram_counts,
                    eng_word_to_cluster, akk_token_to_cluster, honeycomb_sub_zones=honeycomb_sub_zones)
            except Exception as exc:
                print(f"[ERROR] normal sentence {test_idx}: {exc}")
                output = ' '.join(['<gap>'] * len(akk_tokens)).capitalize() + '.'
                routes = []

        ids.append(akk_sent_id)
        translations.append(output)

        # Debug output (first 4 sentences)
        if DEBUG_LEVEL >= 1 and test_idx < 4:
            _log(f"\n[TEST {test_idx}] id={akk_sent_id}")
            _log(f"  Akkadian ({len(akk_tokens)} tokens): {' '.join(akk_tokens[:12])}{'...' if len(akk_tokens)>12 else ''}")
            _log(f"  Constructions detected: {len(constructions)}")

            # Molecular profile
            test_feats = [decompose_token(tok) for tok in akk_tokens]
            test_monomers = [monomer_profile(f) for f in test_feats]
            avg_stab = sum(m['stability'] for m in test_monomers) / max(1, len(test_monomers))
            total_unsat = sum(m['unsatisfied'] for m in test_monomers)
            charges = [m['charge'] for m in test_monomers]
            net_charge = sum(charges)
            lr = long_range_bonds(test_monomers, akk_tokens)
            n_noble = sum(1 for m in test_monomers if m.get('is_noble'))
            n_reactive = len(test_monomers) - n_noble
            _log(f"  [CHEM] Monomers: {len(test_monomers)} | "
                 f"Noble gases: {n_noble} | Reactive: {n_reactive} | "
                 f"Avg stability: {avg_stab:.2f} | Unsatisfied bonds: {total_unsat} | "
                 f"Net charge: {net_charge:+.1f} | Long-range bonds: {len(lr)}")
            if DEBUG_LEVEL >= 2:
                for k, mono in enumerate(test_monomers):
                    tag = "NOBLE" if mono.get('is_noble') else "REACT"
                    _log(f"    [{k}] {mono['token']:<20} {tag:<6} pos={mono['pos']:<8} "
                         f"charge={mono['charge']:+.1f} EN={mono['electronegativity']:.1f} "
                         f"φ={mono['hydrophobicity']:.1f} stab={mono['stability']:.2f} "
                         f"valence={mono['satisfied']}/{mono['valence']}")

            if constructions and DEBUG_LEVEL >= 2:
                for c in constructions[:5]:
                    _log(f"    - {c.get('type')}: {c.get('pattern')} slots={c.get('slots')}")
            _log(f"  Output: {output[:100]}{'...' if len(output)>100 else ''}")
            _log(f"\n  TOKEN ROUTES (Akk -> ... -> Eng):")
            _log(f"  {'-'*68}")
            for akk_tok, route_str, eng_word in routes:
                _log(f"    {akk_tok:<25} -> {eng_word}")
                _log(f"      {route_str}")

    # ── Summary & submission ──────────────────────────────────────
    _log(f"\n{'='*70}")
    _log("TRANSLATION SUMMARY")
    _log(f"{'='*70}")
    _log(f"  Test sentences: {len(translations)}")
    _log(f"  Output sample (first): {translations[0][:120]}{'...' if len(translations[0])>120 else ''}")

    submission_df = pd.DataFrame({'id': ids, 'translation': translations})
    submission_df.to_csv('submission.csv', index=False)

    global _G, _bigram_counts, _eng_word_to_cluster, _akk_token_to_cluster
  
    _G = G
    _bigram_counts = bigram_counts
    _eng_word_to_cluster = eng_word_to_cluster
    _akk_token_to_cluster = akk_token_to_cluster
    _root_mapping = root_to_eng_words
    _cluster_mapping = akk_cluster_to_eng_words
    _test_df = test_df

    elapsed = time.time() - t0
    _log(f"  Submission saved: {len(submission_df)} rows -> submission.csv")
    _log(f"  Total time: {elapsed:.1f}s")
    _log(f"{'='*70}")
    _log("\n[TUNING] To improve, consider:")
    _log("  - BEAM_WIDTH (10): quality-first; decrease for speed")
    _log("  - MAX_LENGTH_FACTOR (1.8): tune if output too short/long")
    _log("  - Increase n_clusters if tokens are too coarse")
    _log("  - COOCC_WEIGHT_THRESHOLD (0.2): lower = more edges, fewer PASSTHROUGH")
    _log("  - Add more SUMERIAN_LOGO entries for logogram coverage")
    _log("  - Populate CAD_DICT from CAD extraction if available")
    _log("  - Populate ONOMASTICON for proper noun handling")
    _log("  - Expand construction_templates for more patterns")
    _log("  - Token routes shown for first 4 test sentences (DEBUG_LEVEL>=1)")


if __name__ == "__main__":
    main()



SANALVARO - Akkadian Translation Pipeline
[CONFIG] DATA_DIR = /kaggle/input/competitions/deep-past-initiative-machine-translation
[CONFIG] DEBUG_LEVEL = 1
[CONFIG] train.csv exists: True
[CONFIG] test.csv exists: True
[CONFIG] eBL_Dictionary.csv exists: True
[EBL] Loaded 10632 entries from eBL_Dictionary.csv
[EBL] OA_QUALITY_PATCH: +55 entries
[EBL] Sample: [('-a i', 'my'), ('-am i', 'me'), ('-iš i', 'to'), ('-ka i', 'you'), ('-ki i', 'you')]
[CAD] Loaded 1221 entries from akkadian_lexiconfromdick.csv
[CAD] Lexicon loaded: 3832 entries
[ONOMASTICON] 11241 spelling variants -> 5904 canonical names
[CLEAN] Training data cleaned: annotations stripped, fractions normalized, quotes straightened, subscripts converted, Ḫ→H
[CLEAN] Test transliterations normalized to match training
[CLEAN] Recovered 3 truncated transliterations from published_texts.csv

[DATA] train: 1561 rows | test: 4 rows
[DATA] Columns: translit=transliteration, trans=translation

[DATA] train: 1561 rows | test: 4 rows
[D

In [23]:
import glob
print(glob.glob('/kaggle/input/**/*.csv', recursive=True))

['/kaggle/input/datasets/alvarochaveste/akkadian-lexicon-clean/akkadian_lexicon_clean.csv', '/kaggle/input/datasets/alvarochaveste/akkylexyfromdic/akkadian_lexiconfromdick.csv', '/kaggle/input/datasets/alvarochaveste/akkylexyfromdic/akkadian_lexiconfromdick1-74.csv', '/kaggle/input/datasets/deeppast/old-assyrian-grammars-and-other-resources/onomasticon.csv', '/kaggle/input/datasets/deeppast/old-assyrian-grammars-and-other-resources/secondary_sources.csv', '/kaggle/input/competitions/deep-past-initiative-machine-translation/sample_submission.csv', '/kaggle/input/competitions/deep-past-initiative-machine-translation/bibliography.csv', '/kaggle/input/competitions/deep-past-initiative-machine-translation/publications.csv', '/kaggle/input/competitions/deep-past-initiative-machine-translation/Sentences_Oare_FirstWord_LinNum.csv', '/kaggle/input/competitions/deep-past-initiative-machine-translation/OA_Lexicon_eBL.csv', '/kaggle/input/competitions/deep-past-initiative-machine-translation/eBL_D

In [24]:
import inspect
import __main__

snapshot = list(vars(__main__).items())

for name, obj in snapshot:
    if callable(obj):
        try:
            src = inspect.getsource(obj)
            if 'COOCC_WEIGHT_THRESHOLD' in src or 'MAX_LENGTH_FACTOR' in src or 'BEAM_WIDTH' in src:
                print(f"\n=== Found in: {name} ===")
                for i, line in enumerate(src.split('\n')):
                    if any(x in line for x in ['COOCC', 'MAX_LENGTH', 'BEAM_WIDTH', 'max_length', 'coocc']):
                        print(f"  line {i}: {line}")
        except:
            pass

for i, row in _test_df.iterrows():
    toks = row['transliteration'].split()
    max_len_old = int(len(toks) * 1.8)
    max_len_new = int(len(toks) * 2.0)
    print(f"TEST {i}: {len(toks)} tokens | old max={max_len_old} | new max={max_len_new}")





for name, obj in snapshot:
    if callable(obj):
        try:
            src = inspect.getsource(obj)
            if 'G.' in src or 'nx.' in src or 'graph' in src.lower() or 'edges' in src.lower():
                print(f"\n=== {name} ===")
                for i, line in enumerate(src.split('\n')):
                    if any(x in line for x in ['G.', 'nx.', 'add_edge', 'edge', 'weight', 'bigram', 'cooc']):
                        print(f"  {i}: {line}")
        except:
            pass


=== Found in: compose_sequence ===
  line 189:     max_len = int(len(akk_tokens) * MAX_LENGTH_FACTOR)
  line 225:         beams = nlargest(BEAM_WIDTH, new_beams, key=lambda x: x[0])

=== Found in: main ===
  line 365:     coocc_edge_count = 0
  line 369:         if weight >= COOCC_WEIGHT_THRESHOLD:
  line 373:                     coocc_edge_count += 1
  line 521:     _log("  - BEAM_WIDTH (10): quality-first; decrease for speed")
  line 522:     _log("  - MAX_LENGTH_FACTOR (1.8): tune if output too short/long")
  line 524:     _log("  - COOCC_WEIGHT_THRESHOLD (0.2): lower = more edges, fewer PASSTHROUGH")
TEST 0: 16 tokens | old max=28 | new max=32
TEST 1: 19 tokens | old max=34 | new max=38
TEST 2: 34 tokens | old max=61 | new max=68
TEST 3: 16 tokens | old max=28 | new max=32

=== TfidfVectorizer ===
  68:         word boundaries; n-grams at the edges of words are padded with space.
  107:         unigrams, ``(1, 2)`` means unigrams and bigrams, and ``(2, 2)`` means
  108:         on

In [25]:
# ═══════════════════════════════════════════════════════════════════════
# CELL A: SILVER AS FILTER (display only — no CSV, no side effects)
#
# Runs RRP gap-fill from pre-RRP snapshots.
# Mendeleev candidates include silver — it competes naturally.
# Silver only appears where neighbor zones already contain it.
# ═══════════════════════════════════════════════════════════════════════

print("=" * 70)
print("CELL A: SILVER FILTER — silver competes naturally in Mendeleev")
print("=" * 70)

def _rrp_silver_filter(best_seq, akk_tokens, akk_feats, G,
                       eng_word_to_cluster, akk_token_to_cluster,
                       eng_pos_to_akk_tok, eng_pos_to_akk_feat,
                       bigram_counts=None):
    """
    RRP gap-filler identical to original EXCEPT:
    Mendeleev does NOT filter any words. Silver competes on merit.
    """
    result = list(best_seq)
    WINDOW = 2
    MIN_CONFIDENCE = 0.3

    eng_to_akk      = eng_pos_to_akk_tok or {}
    eng_to_akk_feat = eng_pos_to_akk_feat or {}

    for i, word in enumerate(result):
        if word != '<gap>':
            continue

        tok = eng_to_akk.get(i)
        if not tok:
            continue

        ac = akk_token_to_cluster.get(tok, -1)
        root = _token_to_root(tok)
        feat = eng_to_akk_feat.get(i) or decompose_token(tok)
        base = feat.get('root') or feat.get('sem_root')
        if ac == -1:
            root_ac = akk_token_to_cluster.get(root, -1) if root else -1
            base_ac = akk_token_to_cluster.get(base, -1) if base else -1
            ac = root_ac if root_ac != -1 else base_ac

        akk_node = f'uc{ac}' if ac != -1 else None

        # Strategy 1: Graph walk (unchanged)
        candidates = {}
        for node_key in [root, base, tok]:
            if node_key and node_key in G:
                for cell in G.successors(node_key):
                    for eng in G.successors(cell):
                        candidates[eng] = candidates.get(eng, 0) + 1

        if candidates and bigram_counts:
            prev_word = result[i-1] if i > 0 else None
            next_word = result[i+1] if i < len(result)-1 else None
            scored = {}
            for eng, count in candidates.items():
                score = count
                if prev_word and (prev_word, eng) in bigram_counts:
                    score += bigram_counts[(prev_word, eng)] * 2
                if next_word and (eng, next_word) in bigram_counts:
                    score += bigram_counts[(eng, next_word)] * 2
                scored[eng] = score
            result[i] = max(scored, key=scored.get)
            continue

        # Strategy 2: Fuzzy (unchanged)
        if not akk_node or akk_node not in G:
            fuzzy_eng, _ = _fuzzy_lookup(tok)
            if fuzzy_eng:
                result[i] = fuzzy_eng
                continue

            # Strategy 2b: MENDELEEV — NO FILTER ON COMMON WORDS
            # Silver competes naturally if it appears in neighbor zones
            neighbor_clusters = []
            neighbor_monomers = []
            for offset in range(-2, 3):
                if offset == 0:
                    continue
                j = i + offset
                if 0 <= j < len(result) and result[j] != '<gap>':
                    nb_ec = eng_word_to_cluster.get(result[j], -1)
                    if nb_ec != -1:
                        neighbor_clusters.append(nb_ec)
                    nb_akk_feat = eng_to_akk_feat.get(j)
                    if nb_akk_feat:
                        neighbor_monomers.append(monomer_profile(nb_akk_feat))

            if not neighbor_clusters:
                continue

            # Valence constraint
            expected_pos = None
            if neighbor_monomers:
                left_mono  = neighbor_monomers[0] if len(neighbor_monomers) > 0 else None
                right_mono = neighbor_monomers[-1] if len(neighbor_monomers) > 1 else None
                if left_mono and left_mono.get('is_verb'):
                    expected_pos = 'NOUN'
                elif left_mono and left_mono['pos'] == 'NOUN' and right_mono and right_mono['pos'] == 'NOUN':
                    expected_pos = 'VERB'
                elif left_mono and left_mono['pos'] == 'NOUN':
                    expected_pos = 'VERB'

            # Collect candidates — NO FILTER, silver can appear
            mendeleev_candidates = {}
            for nc in neighbor_clusters:
                nc_node = f'uc{nc}'
                if nc_node in G:
                    for eng in G.successors(nc_node):
                        if G.nodes.get(eng, {}).get('type') == 'eng_word':
                            if eng == '<gap>':
                                continue  # Only filter gaps, nothing else
                            mendeleev_candidates[eng] = mendeleev_candidates.get(eng, 0) + 1

            if mendeleev_candidates:
                scored = {}
                for cand, freq in mendeleev_candidates.items():
                    score = freq
                    if bigram_counts:
                        prev_w = result[i-1].lower() if i > 0 else None
                        next_w = result[i+1].lower() if i < len(result)-1 else None
                        if prev_w and (prev_w, cand.lower()) in bigram_counts:
                            score += bigram_counts[(prev_w, cand.lower())] * 3
                        if next_w and (cand.lower(), next_w) in bigram_counts:
                            score += bigram_counts[(cand.lower(), next_w)] * 3
                    scored[cand] = score

                if scored:
                    best_cand = max(scored, key=scored.get)
                    if scored[best_cand] >= 1.0:
                        result[i] = best_cand
                        print(f"  [A-MENDELEEV] {tok} -> {best_cand} (score={scored[best_cand]:.1f})")
            continue

        # Strategy 3: Dijkstra (unchanged)
        neighbors = []
        for offset in range(-WINDOW, WINDOW + 1):
            if offset == 0:
                continue
            j = i + offset
            if 0 <= j < len(result) and result[j] != '<gap>':
                neighbors.append(result[j])
        if not neighbors:
            continue
        neighbor_nodes = set()
        for n in neighbors:
            ec = eng_word_to_cluster.get(n, -1)
            if ec != -1:
                neighbor_nodes.add(f'uc{ec}')
            if n in G:
                neighbor_nodes.add(n)
        if not neighbor_nodes:
            continue
        try:
            path_lengths = nx.single_source_dijkstra_path_length(
                G, akk_node,
                weight=lambda u, v, d: -np.log(d.get('weight', 1e-6) + 1e-6),
                cutoff=10)
        except Exception:
            continue
        eng_candidates = {
            n: path_lengths[n] for n in path_lengths
            if G.nodes.get(n, {}).get('type') == 'eng_word'
        }
        if not eng_candidates:
            continue
        scored = []
        for cand_word, dist_score in eng_candidates.items():
            coherence = 0.0
            cand_ec = eng_word_to_cluster.get(cand_word, -1)
            cand_node = f'uc{cand_ec}' if cand_ec != -1 else None
            for nb_node in neighbor_nodes:
                if cand_node and G.has_edge(nb_node, cand_node):
                    coherence += G[nb_node][cand_node].get('weight', 0)
                elif G.has_edge(nb_node, cand_word):
                    coherence += G[nb_node][cand_word].get('weight', 0)
            final_score = np.exp(-abs(dist_score)) + coherence
            scored.append((cand_word, final_score))
        if scored:
            best_cand, best_score = max(scored, key=lambda x: x[1])
            if best_score >= MIN_CONFIDENCE:
                result[i] = best_cand

    return result


# ── Run Cell A ──────────────────────────────────────────────────────
for idx, snap in enumerate(_pre_rrp_snapshots):
    seq = _rrp_silver_filter(
        snap['best_seq'], snap['akk_tokens'], snap['akk_feats'], _G,
        _eng_word_to_cluster, _akk_token_to_cluster,
        snap['eng_pos_to_akk_tok'], snap['eng_pos_to_akk_feat'],
        bigram_counts=_bigram_counts
    )
    seq = _post_process_sequence(seq)
    output = ' '.join(seq).capitalize() + '.'
    gap_count = sum(1 for w in seq if w == '<gap>')
    print(f"\n[FILTER-{idx}] ({gap_count} gaps remaining)")
    print(f"  {output[:150]}{'...' if len(output)>150 else ''}")

print(f"\n{'='*70}")
print("Cell A complete. No files saved.")
print(f"{'='*70}")

CELL A: SILVER FILTER — silver competes naturally in Mendeleev

[FILTER-0] (0 gaps remaining)
  And thus says the colony and kanesh city uc126 to provide and to the trading post and speak the trading stations the statement come city uc126.

[FILTER-1] (0 gaps remaining)
  The word uc126 in adû city this day anybody profit uc126 tin uc126 in do on, kanesh aratia take the colony.

[FILTER-2] (0 gaps remaining)
  Just as our word hear let to there to uc126 give palace officials let return palace and uc126 let still not give uc126 tin as much as here uc126 and u...

[FILTER-3] (0 gaps remaining)
  Our word copy to and to the trading post the trading stations send and let tin the merchants give to uc126 the man be recorded.

Cell A complete. No files saved.


In [26]:
print(repr('aa-qí-il…'))  # look at the actual bytes
tok = 'aa-qí-il…'
print('…' in tok)          # True if U+2026 matches U+2026
print('...' in tok)         # True if it's actually three dots
print([hex(ord(c)) for c in tok[-3:]])  # show exact bytes at end

'aa-qí-il…'
True
False
['0x69', '0x6c', '0x2026']


In [27]:
with open(DATA_DIR / "test.csv", 'r', encoding='utf-8') as f:
    raw = f.read()
    
# Find a line containing au-mì or au-um-au
for line in raw.split('\n'):
    if 'au-m' in line or 'au-um' in line:
        print(repr(line))
        break

'2,332fda50,14,24,ki-ma mup-pì-ni ta-áa-me-a-ni a-ma-kam lu a-na aí-mì-im a-na É.GAL-lim i-dí-in lu té-ra-at É.GAL-lim ú-kà-lim lu na-aí-ma a-dí-ni lá i-dí-in ma-lá KÙ.AN na-áa-ú ni-bi„-it a-aí-im au-um-au ú au-mì a-bi„-au i-na mup-pì-im lu-up-ta-nim-ma ia-tí aí-ip-ri-ni aé-bi„-lá-nim'


In [28]:
import os
for root, dirs, files in os.walk('/kaggle/input/datasets/alvarochaveste/akkylexyfromdic'):
    for f in files:
        print(os.path.join(root, f))

/kaggle/input/datasets/alvarochaveste/akkylexyfromdic/akkadian_lexiconfromdick.csv
/kaggle/input/datasets/alvarochaveste/akkylexyfromdic/akkadian_lexiconfromdick1-74.csv


In [29]:
import glob
print(glob.glob('/kaggle/input/**/*.csv', recursive=True))


['/kaggle/input/datasets/alvarochaveste/akkadian-lexicon-clean/akkadian_lexicon_clean.csv', '/kaggle/input/datasets/alvarochaveste/akkylexyfromdic/akkadian_lexiconfromdick.csv', '/kaggle/input/datasets/alvarochaveste/akkylexyfromdic/akkadian_lexiconfromdick1-74.csv', '/kaggle/input/datasets/deeppast/old-assyrian-grammars-and-other-resources/onomasticon.csv', '/kaggle/input/datasets/deeppast/old-assyrian-grammars-and-other-resources/secondary_sources.csv', '/kaggle/input/competitions/deep-past-initiative-machine-translation/sample_submission.csv', '/kaggle/input/competitions/deep-past-initiative-machine-translation/bibliography.csv', '/kaggle/input/competitions/deep-past-initiative-machine-translation/publications.csv', '/kaggle/input/competitions/deep-past-initiative-machine-translation/Sentences_Oare_FirstWord_LinNum.csv', '/kaggle/input/competitions/deep-past-initiative-machine-translation/OA_Lexicon_eBL.csv', '/kaggle/input/competitions/deep-past-initiative-machine-translation/eBL_D

In [30]:
import pandas as pd
ono_df = pd.read_csv('/kaggle/input/datasets/deeppast/old-assyrian-grammars-and-other-resources/onomasticon.csv')
print(ono_df.head(10))
print(ono_df.columns.tolist())

                   Name Duplicate Spellings_semicolon_separated Aliases
0          {d}PA-UR.SAG     False                  {d}PA-UR.SAG     NaN
1           {d}UTU-bani     False                  {d}UTU-ba-ni     NaN
2            {d}UTU-GAL     False                    {d}UTU-GAL     NaN
3  A-<gap>-ku-<big_gap>     False          A-<gap>-ku-<big_gap>     NaN
4            A-áš-be-el     False                    A-áš-be-el     NaN
5        A-bi4-ra-<gap>     False                A-bi4-ra-<gap>     NaN
6               A-dí-bu     False                       A-dí-bu     NaN
7             A-ha-tí-e     False                     A-ha-tí-e     NaN
8               A-na-ar     False                       A-na-ar     NaN
9        A-na-na-ar-tim     False                A-na-na-ar-tim     NaN
['Name', 'Duplicate', 'Spellings_semicolon_separated', 'Aliases']
